In [36]:
import torch 
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
best_score = -999.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
alpha = 15
beta = 170
delta = 0.01
dt = 0.5
T = 20
ego_v_d = torch.tensor(5.0, device=device)
L = 4
u_min, u_max = -5.0, 5.0
K = 3

# 匝道方向
theta_deg = 20.0
theta = np.deg2rad(theta_deg)
# 匝道的方向向量
d_np = np.array([np.cos(theta), np.sin(theta)])
d_np = d_np / (np.linalg.norm(d_np) + 1e-12)
d_vec = torch.tensor(d_np, device=device, dtype=torch.float32)

n = np.array([-np.sin(theta), np.cos(theta)])  
# merge point 
#x_merge = 0.0

# main road center point
y_merge_val = -10.5
y_merge = torch.tensor(y_merge_val, device=device, dtype=torch.float32)

# merge point position
p_merge_np = np.array([0.0, -10.5])
p_merge = torch.tensor(p_merge_np, device=device, dtype=torch.float32)

lane_th = 0.6

In [2]:
def signed_t_from_merge_torch(p_x, p_y):
    # p = [p_x, p_y]
    # return dot(p - p_merge, d)
    px = p_x - p_merge[0]
    py = p_y - p_merge[1]
    return px * d_vec[0] + py * d_vec[1]

ZONE_MIN = -40.0
ZONE_MAX = -25.0

def in_collision_zone_torch(p_x, x2):
    # main lane y ≈ y_merge, ego x 在 [-zone_x, zone_x]
    ego_in  = (p_x >= ZONE_MIN) & (p_x <= ZONE_MAX)
    car2_in = (x2  >= ZONE_MIN) & (x2  <= ZONE_MAX)
    return ego_in & car2_in

def ego_is_slowing_torch(u1, v1, v1_d, a_slow_th=0.3, v_slow_ratio=0.80):
    cond_brake = u1 < -a_slow_th # ego 明显刹车
    cond_slow  = v1 < (v1_d * v_slow_ratio) # ego速度低于目标
    return cond_brake | cond_slow # ture：ego让速； false：ego没有让速

In [3]:
def compute_leaders_torch(
    x_cars, v_cars, style_ids,
    ego_x, ego_v, ego_on_main,
    lookahead: float = 60.0,
):
    """
    x_cars, v_cars, style_ids: [N, K]
    ego_x, ego_v: [N]
    ego_on_main: [N] bool 或 list[bool]

    输出:
      leader_x, leader_v: [N, K]
      has_leader: [N, K] bool
    """
    device = x_cars.device
    dtype  = x_cars.dtype
    N, K   = x_cars.shape

    # ---------- 1) 主路车之间互相找前车 ----------
    # x_i[n,i,1] = x_cars[n,i]
    # x_j[n,1,j] = x_cars[n,j]
    # dx[n,i,j]  = x_j - x_i
    x_i = x_cars.unsqueeze(2)   # [N, K, 1]
    x_j = x_cars.unsqueeze(1)   # [N, 1, K]
    dx  = x_j - x_i             # [N, K, K]

    # 自己不能当自己的 leader
    eye = torch.eye(K, device=device, dtype=torch.bool).unsqueeze(0)  # [1,K,K]

    # 有效候选：同一 episode 内，前方 (dx>0)，且在 lookahead 内
    valid_car = (~eye) & (dx > 0.0) & (dx < lookahead)   # [N,K,K]

    big = torch.tensor(1e9, device=device, dtype=dtype)

    # 把无效候选的 dx 设成 big，min 的时候就会被忽略
    dx_masked = torch.where(valid_car, dx, big)          # [N,K,K]

    # 对 j 维 (候选车) 求最小值：得到最近前车的距离和 index
    min_dx_car, idx_car = dx_masked.min(dim=2)           # [N,K], [N,K]
    has_car_leader = min_dx_car < (big * 0.5)            # [N,K] 是否存在车-leader

    idx_car = idx_car.clamp(0, K-1)
    leader_x_car = torch.gather(x_cars, 1, idx_car)      # [N,K]
    leader_v_car = torch.gather(v_cars, 1, idx_car)      # [N,K]

    # ---------- 2) ego 作为候选 ----------
    ego_x_exp = ego_x.unsqueeze(1).expand(N, K)          # [N,K]
    ego_v_exp = ego_v.unsqueeze(1).expand(N, K)          # [N,K]

    dx_ego = ego_x_exp - x_cars                         # [N,K]

    if isinstance(ego_on_main, torch.Tensor):
        # 处理不同形状的 ego_on_main
        if ego_on_main.dim() == 0:  # 标量
            ego_main_mask = ego_on_main.expand(N, K)  # [N,K]
        elif ego_on_main.dim() == 1:  # [N] 或 [1]
            if ego_on_main.shape[0] == 1 and N > 1:
                # [1] -> [N] -> [N,K]
                ego_main_mask = ego_on_main.expand(N).unsqueeze(1).expand(N, K)
            else:
                # [N] -> [N,K]
                ego_main_mask = ego_on_main.unsqueeze(1).expand(N, K)
        else:
            # 已经是 [N,K] 或其他形状，直接使用
            ego_main_mask = ego_on_main.expand(N, K) if ego_on_main.shape != (N, K) else ego_on_main
    else:
        # 非 tensor，转换为 tensor
        ego_on_main_tensor = torch.tensor(ego_on_main, device=device)
        if ego_on_main_tensor.dim() == 0:
            ego_main_mask = ego_on_main_tensor.expand(N, K)
        else:
            ego_main_mask = ego_on_main_tensor.unsqueeze(1).expand(N, K)

    # ego 必须在主路上，而且在前方、在 lookahead 之内
    valid_ego = ego_main_mask & (dx_ego > 0.0) & (dx_ego < lookahead)   # [N,K]

    # 如果 ego 是候选，并且：
    #   - 原来没有车-leader，或者
    #   - ego 比车-leader 更近
    # 那么该位置就应该用 ego 当 leader
    use_ego = valid_ego & ((~has_car_leader) | (dx_ego < min_dx_car))

    # ---------- 3) 综合车-leader 和 ego-leader ----------
    has_leader = has_car_leader | use_ego               # [N,K]

    # 先用车-leader 初始化
    leader_x = leader_x_car.clone()
    leader_v = leader_v_car.clone()

    # 在需要使用 ego 的位置上，用 ego 的 x/v 覆盖
    leader_x = torch.where(use_ego, ego_x_exp, leader_x)
    leader_v = torch.where(use_ego, ego_v_exp, leader_v)

    return leader_x, leader_v, has_leader


In [4]:
def idm_acc_torch(v, v_lead, gap, cfg, eps=1e-6):
    # v, v_lead, gap: [N]
    v  = torch.clamp(v, min=0.0)
    gap = torch.clamp(gap, min=0.1)

    v0    = cfg["v0"]
    T_h   = cfg["T"]
    s0    = cfg["s0"]
    a_max = cfg["a"]
    b_comf= cfg["b"]
    delta = cfg.get("delta", 4.0)

    v0 = torch.as_tensor(v0, device=v.device, dtype=v.dtype)
    T_h = torch.as_tensor(T_h, device=v.device, dtype=v.dtype)
    s0 = torch.as_tensor(s0, device=v.device, dtype=v.dtype)
    a_max = torch.as_tensor(a_max, device=v.device, dtype=v.dtype)
    b_comf = torch.as_tensor(b_comf, device=v.device, dtype=v.dtype)
    delta_p = torch.as_tensor(delta, device=v.device, dtype=v.dtype)

    dv = v - v_lead
    denom = 2.0 * torch.sqrt(torch.clamp(a_max * b_comf, min=eps))
    s_star = s0 + v * T_h + (v * dv) / denom
    s_star = torch.maximum(s0, s_star)

    acc = a_max * (1.0 - (v / torch.clamp(v0, min=1e-3))**delta_p - (s_star / gap)**2)
    return acc

cfg_follow_style0 = dict(   
    # aggressive, gap small
    v0=10.0,  
    T= 2.0,    
    s0=2.0,
    a=2.5,
    b=3.0,
    delta=4.0,
)

cfg_follow_style1 = dict(   
    # reactive, gap big
    v0=8.0,  
    T=4.5,    # headway 小
    s0=3.5,
    a=0.8,
    b=2.0,
    delta=4.0,
)

cfg_follow_style2 = dict(   # stop-line queue
    v0=6.0,
    T=1.4,
    s0=2.5,
    a=1.2,
    b=2.0,
    delta=4.0,
)

cfg_fast  = dict(v0=12.0, T=1.5, s0=1.5, a=3.0, b=4.0, delta=4.0)  # 改进：增加T和s0以降低碰撞率
cfg_yield = dict(v0=6.0, T=1.6, s0=2.5, a=1.2, b=2.0, delta=4.0)
cfg_fast_style1  = dict(v0=10.0, T=1.5, s0=2.0, a=2.0, b=2.0, delta=4.0)  



STYLE_AGGRESSIVE = 0
STYLE_REACTIVE   = 1 
STYLE_YIELD      = 2
style_list = [STYLE_AGGRESSIVE, STYLE_REACTIVE, STYLE_YIELD]
M_styles   = len(style_list)

u_min, u_max = -5.0, 5.0

def step_car2_style_torch(x2, v2, p_x, p_y, v1, u1, v1_d, style_id):
    """
    x2, v2, p_x, p_y, v1, u1, style_id: 可以是 [N] 或 [1]
    返回: a2, 同形状
    """
    STOP_X = -25.0

    # ========== 通用：zone / ego_slowing ==========
    # in_zone     = in_collision_zone_torch(p_x, x2)
    ego_slowing = ego_is_slowing_torch(u1, v1, v1_d)

    # ---------- 基础 IDM 加速度 ----------
    # 区域外的巡航（比较温和）
    a2_free = idm_acc_torch(
        v2,
        v2,
        torch.full_like(v2, 1e9),
        cfg_yield
    )

    # “抢行”版本（更激进）
    a2_fast = idm_acc_torch(
        v2,
        v2,
        torch.full_like(v2, 1e9),
        cfg_fast
    )

    a2_fast_style1 = idm_acc_torch(
        v2,
        v2,
        torch.full_like(v2, 1e9),
        cfg_fast_style1 
    )
    # =========================================================================
    # ========== style 0: AGGRESSIVE ==========
    # 简单：不管 zone，始终用 fast
    a2_s0 = a2_fast

    # ========== style 1: REACTIVE ==========
    # 改进逻辑：根据 ego 的行为反应，更好地区分 Style 1 和 Style 2
    # 关键：Style 1 应该更"主动"，即使 ego 加速，主路车也不应该总是让行
    STOP_X = -25.0
    CLEAR_X = 5.0  # ego通过merge的判据
    CLOSE_DISTANCE = 15.0  # 主路车和 ego 的接近距离阈值
    VERY_CLOSE_DISTANCE = 10.0
    
    # ego已经进入-25
    ego_committed = (p_x >= STOP_X)
    ego_cleared = (p_x > CLEAR_X)  # ego已经通过merge点
    
    # ⭐ 改进：使用 ego_front_y 判断 ego 是否在主路上（与 potentialFunction_update_merge_Ncars 保持一致）
    # 计算车头位置（沿车辆前进方向，车头在中心前方 L/2 米）
    ego_front_y = p_y + (L / 2.0) * d_vec[1]  # [N] 车头的 y 坐标
    
    # 判断条件：车头的 y 坐标 > -14（刚好进入匝道口）
    MERGE_Y_FRONT_THRESHOLD = -14.0
    ego_on_main = ego_front_y > MERGE_Y_FRONT_THRESHOLD  # [N] bool
    
    # ⭐ 改进：相对位置判定（考虑 y 坐标）
    # 如果 ego 还在合并车道上（ego_front_y <= -14），主路车总是在前面
    # 如果 ego 在主路上（ego_front_y > -14），使用 x 坐标判断前后关系
    i_am_ahead = (x2 > p_x) | ((torch.abs(x2 - p_x) < 5.0) & (~ego_on_main))  # 主路车在ego前面，或x接近且ego还在合并车道
    i_am_behind = (x2 < p_x) & ego_on_main  # 主路车在ego后面，且ego在主路上
    
    
    # 计算主路车到 ego 的距离（当主路车在 ego 后面时）
    distance_to_ego = p_x - x2  # 如果 x2 < p_x，distance_to_ego > 0
    behind_valid = i_am_behind & (distance_to_ego > 0.0)
    close_to_ego = (distance_to_ego > 0.0) & (distance_to_ego < CLOSE_DISTANCE)  # 主路车在 ego 后面且距离较近
    very_close_to_ego = (distance_to_ego > 0.0) & (distance_to_ego < VERY_CLOSE_DISTANCE)
    mid_close_to_ego  = close_to_ego & (~very_close_to_ego) 
    far_from_ego      = behind_valid & (distance_to_ego >= CLOSE_DISTANCE)  # 主路车在 ego 后面但距离较远
    
    # 改进逻辑：让 Style 1 更"主动"，防止 Style 2 的策略（总是加速）在 Style 1 上也能工作
    # - 如果ego已经进入-25，且还没通过CLEAR_X
    #   - 如果主路车在ego前面 -> 加速冲（不让行）
    #   - 如果主路车在ego后面：
    #     * 如果 ego 在减速（让行），且主路车接近 ego -> 加速通过（利用让行机会）
    #     * 如果 ego 在加速（抢行），且主路车接近 ego -> 减速至0（完全让行）
    #     * 如果主路车距离 ego 较远 -> 轻微减速（不完全让行，保持博弈）
    # - 如果ego已经通过CLEAR_X -> 恢复自由巡航
    
    # 减速至0的逻辑：使用yield_speed=0.0作为目标速度（完全让行）
    cfg_yield_very_low = dict(cfg_yield)
    cfg_yield_very_low["v0"] = 0.5  # 目标速度0.0（完全让行）
    a2_slow_to_0 = idm_acc_torch(
        v2,
        v2,
        torch.full_like(v2, 1e9),
        cfg_yield_very_low
    )
    
    # 轻微减速的逻辑：使用yield_speed=1.5作为目标速度（不完全让行，保持博弈）
    # 减速至1-2，而不是完全停止，这样可以保持博弈，不完全让行
    cfg_yield_mild = dict(cfg_yield)
    cfg_yield_mild["v0"] = 2.0  # 目标速度2.0（不完全让行，保持博弈）
    a2_slow_mild = idm_acc_torch(
        v2,
        v2,
        torch.full_like(v2, 1e9),
        cfg_yield_mild
    )
    
    # 决策逻辑
    # 情况1: ego已经进入-25，且主路车在ego前面 -> 加速冲（不让行）
    logic_ahead_clear = ego_committed & (~ego_cleared) & i_am_ahead
    
    # 情况2a: ego已经进入-25，且主路车在ego后面，且 ego 在减速（让行），且主路车接近 ego -> 加速通过
    logic_behind_rush = ego_committed & (~ego_cleared) & i_am_behind & ego_slowing & very_close_to_ego
    
    # 情况2b: ego已经进入-25，且主路车在ego后面，且 ego 在加速（抢行），且主路车接近 ego -> 减速至0（完全让行）
    logic_behind_yield_close = ego_committed & (~ego_cleared) & i_am_behind & (~ego_slowing) & very_close_to_ego
    
    logic_behind_yield_mid = ego_committed & (~ego_cleared) & mid_close_to_ego

    # 情况2c: ego已经进入-25，且主路车在ego后面，且主路车距离 ego 较远 -> 轻微减速（不完全让行）
    logic_behind_yield_far = ego_committed & (~ego_cleared) & far_from_ego
    
    # 情况3: ego还没进入-25，或已经通过CLEAR_X -> 自由巡航
    # (其他情况)
    
    # 执行决策
    a2_s1 = a2_free.clone()  # 默认自由巡航
    a2_s1 = torch.where(logic_ahead_clear, a2_fast_style1, a2_s1)  # 在ego前面 -> 加速冲
    a2_s1 = torch.where(logic_behind_rush, a2_fast_style1, a2_s1)  # ⭐ ego 减速且主路车接近 ego -> 加速通过
    a2_s1 = torch.where(logic_behind_yield_close, a2_slow_to_0, a2_s1)  # ⭐ ego 加速且主路车接近 ego -> 减速至0（完全让行）
    a2_s1 = torch.where(logic_behind_yield_mid, a2_slow_mild, a2_s1)
    a2_s1 = torch.where(logic_behind_yield_far, a2_slow_mild, a2_s1)  # ⭐ 主路车距离 ego 较远 -> 轻微减速（不完全让行）

    # ========== style 2: YIELD at STOP_X ==========
  
    CLEAR_X  = 5.0      
    STOP_EPS = 2.0      

    before_stop = (x2 < STOP_X - STOP_EPS)               
    at_stop     = (x2 >= STOP_X - STOP_EPS) & (x2 <= STOP_X + STOP_EPS)
    after_stop  = (x2 >  STOP_X + STOP_EPS)              
    ego_cleared = (p_x > CLEAR_X)

    gap_to_stop = (STOP_X - x2).clamp(min=0.1)
    a2_toward_stop = idm_acc_torch(
        v2,
        torch.zeros_like(v2),                             
        gap_to_stop,
        cfg_yield
    )

    a2_wait = torch.where(
        v2 > 0,
        torch.full_like(v2, u_min),                       
        torch.zeros_like(v2)                              
    )

    a2_resume = idm_acc_torch(
        v2,
        v2,
        torch.full_like(v2, 1e9),
        cfg_yield
    )

    a2_s2 = torch.zeros_like(v2)
    a2_s2 = torch.where(before_stop, a2_toward_stop, a2_s2)
    wait_zone = at_stop & (~ego_cleared)
    a2_s2 = torch.where(wait_zone, a2_wait, a2_s2)
    resume_zone = ego_cleared | after_stop
    a2_s2 = torch.where(resume_zone, a2_resume, a2_s2)

    # ========== 按 style_id 选择 ==========
    style_id = style_id.long()
    if style_id.dim() == 0:
        style_id = style_id.expand_as(v2)

    a2 = torch.where(
        style_id == STYLE_AGGRESSIVE, a2_s0,
        torch.where(style_id == STYLE_REACTIVE, a2_s1, a2_s2)
    )

    return torch.clamp(a2, u_min, u_max)



In [5]:
def step_vehicle_Ncars_torch(
    x_cars, v_cars,                 # [N, K]
    leader_x, leader_v, has_leader, # [N, K]
    p_x_ego, p_y_ego,               # [N] 或标量
    v_ego, u_ego_t, v_ego_d,        # [N] 或标量
    style_ids                       # [N, K]
):
    """
    Batched 版本（N episodes, K cars each episode）
    """
    device = x_cars.device
    dtype  = x_cars.dtype
    N, K   = x_cars.shape

    # --------- 保证 ego 相关量都是 tensor，兼容 float 标量 ---------
    if not torch.is_tensor(p_x_ego):
        p_x_ego = torch.tensor(p_x_ego, device=device, dtype=dtype)
    if not torch.is_tensor(p_y_ego):
        p_y_ego = torch.tensor(p_y_ego, device=device, dtype=dtype)
    if not torch.is_tensor(v_ego):
        v_ego = torch.tensor(v_ego, device=device, dtype=dtype)
    if not torch.is_tensor(u_ego_t):
        u_ego_t = torch.tensor(u_ego_t, device=device, dtype=dtype)
    if not torch.is_tensor(v_ego_d):
        v_ego_d = torch.tensor(v_ego_d, device=device, dtype=dtype)

    # 展平成一维
    x_flat       = x_cars.reshape(-1)          # [N*K]
    v_flat       = v_cars.reshape(-1)          # [N*K]
    leader_x_f   = leader_x.reshape(-1)        # [N*K]
    leader_v_f   = leader_v.reshape(-1)        # [N*K]
    has_leader_f = has_leader.reshape(-1)      # [N*K] bool
    style_f      = style_ids.reshape(-1).long()

    # ego 参数 broadcast 到 [N,K] 再 flatten
    p_x_flat   = p_x_ego.unsqueeze(1).expand(N, K).reshape(-1)
    p_y_flat   = p_y_ego.unsqueeze(1).expand(N, K).reshape(-1)  # ⭐ 新增
    v_ego_flat = v_ego.unsqueeze(1).expand(N, K).reshape(-1)
    u_ego_flat = u_ego_t.unsqueeze(1).expand(N, K).reshape(-1)

    if v_ego_d.dim() == 0:
        v_d_flat = v_ego_d.expand(N * K)
    elif v_ego_d.dim() == 1:
        v_d_flat = v_ego_d.unsqueeze(1).expand(N, K).reshape(-1)
    else:
        v_d_flat = v_ego_d.reshape(-1)

    a_flat = torch.zeros_like(x_flat)

    # ============================================================
    # 1) follower：按 style0/1/2 做 IDM 跟车
    # ============================================================
    follow_mask = has_leader_f
    if follow_mask.any():
        idx_f = follow_mask.nonzero(as_tuple=True)[0]
        gap_f = (leader_x_f[idx_f] - x_flat[idx_f] - L).clamp(min=0.1)
        v_f   = v_flat[idx_f]
        vL_f  = leader_v_f[idx_f]
        st    = style_f[idx_f]

        a_follow = torch.zeros_like(v_f)
        # 这里你可以根据需要调用不同的 cfg
        # ... (此处保持你原有的 cfg 逻辑，或者简化成统一函数) ...
        m0 = (st == STYLE_AGGRESSIVE)
        m1 = (st == STYLE_REACTIVE)
        m2 = (st == STYLE_YIELD)
        if m0.any():
            ii = m0.nonzero(as_tuple=True)[0]
            a_follow[ii] = idm_acc_torch(v=v_f[ii], v_lead=vL_f[ii], gap=gap_f[ii], cfg=cfg_follow_style0)
        if m1.any():
            ii = m1.nonzero(as_tuple=True)[0]
            a_follow[ii] = idm_acc_torch(v=v_f[ii], v_lead=vL_f[ii], gap=gap_f[ii], cfg=cfg_follow_style1)
        if m2.any():
            ii = m2.nonzero(as_tuple=True)[0]
            a_follow[ii] = idm_acc_torch(v=v_f[ii], v_lead=vL_f[ii], gap=gap_f[ii], cfg=cfg_follow_style2)

        a_flat[idx_f] = a_follow

    # ============================================================
    # 1.5) Style 0：AGGRESSIVE - 不管 ego，只有 ego.x > 5 才跟 ego
    # ============================================================
    EGO_CLEAR_X_FOR_S0 = 5.0  # 只有 ego 完全过了这个点才跟
    
    style0_mask_2d = (style_ids == STYLE_AGGRESSIVE)
    
    # ego 还没完全过（ego.x <= 5），Style 0 不把 ego 当 leader
    ego_not_clear_s0 = (p_x_ego <= EGO_CLEAR_X_FOR_S0).unsqueeze(1).expand(N, K)
    
    # Style 0 的车需要覆盖
    style0_override_2d = style0_mask_2d & ego_not_clear_s0
    style0_override_f = style0_override_2d.reshape(-1)
    
    if style0_override_f.any():
        idx_s0 = style0_override_f.nonzero(as_tuple=True)[0]
        
        v_s0 = v_flat[idx_s0]
        x_s0 = x_flat[idx_s0]
        
        # 检查 leader 是不是 ego（通过比较位置）
        has_leader_s0 = has_leader_f[idx_s0]
        leader_x_s0 = leader_x_f[idx_s0]
        leader_v_s0 = leader_v_f[idx_s0]
        p_x_s0 = p_x_flat[idx_s0]
        
        # leader 和 ego 位置几乎相同 → leader 是 ego
        leader_is_ego = (torch.abs(leader_x_s0 - p_x_s0) < 0.5)
        
        # 有效的车 leader：有 leader 且 leader 不是 ego
        has_car_leader = has_leader_s0 & (~leader_is_ego)
        
        a_s0 = torch.zeros_like(v_s0)
        
        # 情况 1：有其他主路车在前面，跟车
        if has_car_leader.any():
            ii = has_car_leader.nonzero(as_tuple=True)[0]
            gap = (leader_x_s0[ii] - x_s0[ii] - L).clamp(min=0.1)
            a_s0[ii] = idm_acc_torch(v_s0[ii], leader_v_s0[ii], gap, cfg_fast)
        
        # 情况 2：没有其他主路车在前面，自由 fast 前进
        no_car_leader = ~has_car_leader
        if no_car_leader.any():
            ii = no_car_leader.nonzero(as_tuple=True)[0]
            a_s0[ii] = idm_acc_torch(
                v_s0[ii], 
                v_s0[ii], 
                torch.full_like(v_s0[ii], 1e9), 
                cfg_fast
            )
        
        a_flat[idx_s0] = torch.clamp(a_s0, u_min, u_max)

    # ============================================================
    # 2) style1：博弈逻辑 (覆盖)
    # ============================================================
    # #### 修改点 2：扩大博弈区域 ####
    # 原来是 [-40, -25]，现在把上界拉到 10.0 甚至更远
    # 这样 Ego 只要在路口附近(哪怕刚过线)，Style 1 的车依然会保持“让行”态，直到彻底远离。
    
    CONFLICT_X_MIN = -40.0
    CONFLICT_X_MAX = 10.0   # <--- 修改：从 -25.0 改为 10.0 (或者更大)
    
    EGO_X_MIN      = -35.0
    EGO_X_MAX      = 10.0   # <--- 修改：从 -25.0 改为 10.0，让博弈延续到 merge 后


    car_in_conflict = (x_cars >= CONFLICT_X_MIN) & (x_cars <= CONFLICT_X_MAX)    # [N,K]
    ego_in_conflict = (p_x_ego >= EGO_X_MIN) & (p_x_ego <= EGO_X_MAX)            # [N]
    ego_in_conf_2d  = ego_in_conflict.unsqueeze(1).expand(N, K)                  # [N,K]

    style1_mask_2d  = (style_ids == STYLE_REACTIVE)
    
    # 只有当 Ego 和 Car 都在区域内，才触发博弈
    game1_mask_2d   = style1_mask_2d & ego_in_conf_2d & car_in_conflict          # [N,K]
    game1_mask_f    = game1_mask_2d.reshape(-1)

    if game1_mask_f.any():
        idx_g1 = game1_mask_f.nonzero(as_tuple=True)[0]
        a_game1 = step_car2_style_torch(
            x2       = x_flat[idx_g1],
            v2       = v_flat[idx_g1],
            p_x      = p_x_flat[idx_g1],
            p_y      = p_y_flat[idx_g1],  # ⭐ 新增
            v1       = v_ego_flat[idx_g1],
            u1       = u_ego_flat[idx_g1],
            v1_d     = v_d_flat[idx_g1],
            style_id = style_f[idx_g1],   # 全是 style1
        )
        a_flat[idx_g1] = a_game1

    # ============================================================
    # 3) style2：stopline car (覆盖)
    # ============================================================
    STOP_X   = -25.0
    BRAKE_DIST = 20.0
    STOP_EPS   = 1.0

    # 候选：style2 在 stopline 前/附近
    in_brake_window = (x_cars >= (STOP_X - BRAKE_DIST)) & (x_cars <= (STOP_X + STOP_EPS))
    candidate2 = (style_ids == STYLE_YIELD) & in_brake_window  # [N,K]

    # 每行选 x 最大的候选
    x_masked2 = torch.where(candidate2, x_cars, torch.full_like(x_cars, -1e9))
    _, idx_col2 = x_masked2.max(dim=1)              # [N]
    has_stop2 = candidate2.any(dim=1)               # [N]

    stop2_mask_2d = torch.zeros_like(candidate2, dtype=torch.bool)  # [N,K]
    stop2_mask_2d[has_stop2, idx_col2[has_stop2]] = True
    stop2_mask_f = stop2_mask_2d.reshape(-1)

    if stop2_mask_f.any():
        idx_g2 = stop2_mask_f.nonzero(as_tuple=True)[0]
        a_game2 = step_car2_style_torch(
            x2       = x_flat[idx_g2],
            v2       = v_flat[idx_g2],
            p_x      = p_x_flat[idx_g2],
            p_y      = p_y_flat[idx_g2],  # ⭐ 新增
            v1       = v_ego_flat[idx_g2],
            u1       = u_ego_flat[idx_g2],
            v1_d     = v_d_flat[idx_g2],
            style_id = torch.full_like(v_flat[idx_g2], STYLE_YIELD, dtype=torch.long),
        )
        a_flat[idx_g2] = a_game2

    # ============================================================
    # 4) 其它 no-leader 且未被覆盖：自由巡航
    # ============================================================
    no_leader_f = ~has_leader_f
    covered_f = game1_mask_f | stop2_mask_f
    free_mask = no_leader_f & (~covered_f)

    if free_mask.any():
        idx_free = free_mask.nonzero(as_tuple=True)[0]

        v_cur = v_flat[idx_free]
        st    = style_f[idx_free]
        gap_big = torch.full_like(v_cur, 1e9)
        a_free = torch.zeros_like(v_cur)

        m0 = (st == STYLE_AGGRESSIVE)
        m1 = (st == STYLE_REACTIVE)
        m2 = (st == STYLE_YIELD)

        if m0.any():
            ii = m0.nonzero(as_tuple=True)[0]
            a_free[ii] = idm_acc_torch(v=v_cur[ii], v_lead=v_cur[ii], gap=gap_big[ii], cfg=cfg_follow_style0)
        if m1.any():
            ii = m1.nonzero(as_tuple=True)[0]
            a_free[ii] = idm_acc_torch(v=v_cur[ii], v_lead=v_cur[ii], gap=gap_big[ii], cfg=cfg_follow_style1)
        if m2.any():
            ii = m2.nonzero(as_tuple=True)[0]
            a_free[ii] = idm_acc_torch(v=v_cur[ii], v_lead=v_cur[ii], gap=gap_big[ii], cfg=cfg_follow_style2)

        a_flat[idx_free] = torch.clamp(a_free, u_min, u_max)

    return a_flat.view(N, K)




In [6]:
# potentialFunction_update_merge_Ncars

def potentialFunction_update_merge_Ncars(
        u_batch_seq,                # [N, T]  ego 的 a 序列
        ego_x0, ego_y0, ego_v0,     # [N]
        x_cars0, v_cars0,           # [N, K]  背景车初始位置/速度
        style_ids,                  # [N, K]  每辆车的 style (0/1/2)
        v_ego_d,                    # ego 期望速度
        alpha, beta, delta,
        T, dt,

):
    device = u_batch_seq.device
    N, K = x_cars0.shape
    
    # init
    v1t   = ego_v0.clone()      # [N]
    p_x   = ego_x0.clone()      # [N]
    p_y   = ego_y0.clone()      # [N]
    x_cars= x_cars0.clone()   # [N, K]
    v_cars= v_cars0.clone()   # [N, K]

    if not torch.is_tensor(v_ego_d):
        v_ego_d = torch.full((N,), float(v_ego_d),
                             device=device,
                             dtype=ego_v0.dtype)
    elif v_ego_d.dim() == 0:
        v_ego_d = v_ego_d.expand(N)

    self_cost  = torch.zeros_like(v1t)  # [N]
    inter_cost = torch.zeros_like(v1t)  # [N]
    s_buffer = 8.0 # 在距离 merge 点 8 米之内代表conflict

    for t in range(T):
        u1_t = u_batch_seq[:, t]          # [N]

        # ego dynamics
        v1t = v1t + u1_t * dt             # [N] 允许负速度，不 clip
        step_len = v1t * dt               # [N]

        s_old = signed_t_from_merge_torch(p_x, p_y) # [N]
        on_ramp = s_old < 0.0

        # 这一小步是否跨过 merge 点
        cross_merge = on_ramp & (step_len > -s_old)

        # 1) 纯匝道：未跨 merge
        move_ramp = on_ramp & (~cross_merge)
        p_x = torch.where(move_ramp, p_x + d_vec[0] * step_len, p_x)
        p_y = torch.where(move_ramp, p_y + d_vec[1] * step_len, p_y)

        # 2) 跨 merge：先走到 merge，再沿 x 走剩余
        # 剩余距离：step_len + s_old
        remain = step_len + s_old
        # 先加到 merge 点
        p_x = torch.where(cross_merge, p_x + d_vec[0] * (-s_old), p_x)
        p_y = torch.where(cross_merge, p_y + d_vec[1] * (-s_old), p_y)
        # 再沿 x 方向走 remain
        p_x = torch.where(cross_merge, p_x + remain, p_x)
        # p_y 在 main lane 就保持不变（已经是 near y_merge 了）

         # 3) 已在主路：s_old >= 0 时
        on_main = ~on_ramp   # 注意这里用的是旧的 s_old
        p_x = torch.where(on_main, p_x + step_len, p_x)
        # p_y 不动


        # main road cars -- 先确认leader才能算速度，因为leader和不是leader的规则并不相同
        

        # ⭐ 改进：基于车头位置判断 ego_on_main
        # 计算车头位置（沿车辆前进方向，车头在中心前方 L/2 米）
        # d_vec 是合并车道方向向量，d_vec[0] 是 x 方向，d_vec[1] 是 y 方向
        ego_front_y = p_y + (L / 2.0) * d_vec[1]  # [N] 车头的 y 坐标
        
        # 判断条件：车头的 y 坐标 > -14（刚好进入匝道口）
        MERGE_Y_FRONT_THRESHOLD = -14.0
        ego_on_main = ego_front_y > MERGE_Y_FRONT_THRESHOLD  # [N] bool
        
        # 保留 s_now 用于后续的 cost 计算
        s_now = signed_t_from_merge_torch(p_x, p_y)

        leader_x, leader_v, has_leader = compute_leaders_torch(
            x_cars, v_cars, style_ids, p_x, v1t, ego_on_main, lookahead = 60
        )

        a_cars_t = step_vehicle_Ncars_torch(
            x_cars, v_cars, leader_x, leader_v, has_leader, p_x, p_y, v1t, u1_t, v_ego_d, style_ids  # ⭐ 添加 p_y
        )
        v_cars = torch.clamp(v_cars + a_cars_t * dt, min=0.0)
        x_cars = x_cars + v_cars * dt

        # cost
        # self cost
        self_cost = self_cost + ((v1t - v_ego_d)/v_ego_d) ** 2

        # inter cost
        dx = p_x.unsqueeze(1) - x_cars          # [N,K]
        dy = p_y.unsqueeze(1) - y_merge         # y_merge 标量 tensor
        dist2 = dx * dx + dy * dy               # [N,K]

        
        inter_mask = (s_now > -s_buffer).float()
        repulsion = torch.sum(1.0 / (dist2 + delta), dim=1)
        inter_cost = inter_cost + inter_mask * repulsion

    Total_cost = alpha * self_cost + beta * inter_cost
    return Total_cost




In [7]:
def get_stratified_tensor(N, min_val, max_val, device):
    total_dist = max_val - min_val
    grid_base = torch.linspace(min_val, max_val - (total_dist / N), N, device=device)
    noise = torch.rand(N, device=device) * (total_dist / N)
    data = grid_base + noise
    return data[torch.randperm(N, device=device)]

In [8]:

STOP_X = -25
def sample_merge_joint_space(
    N,
    device,
    v1_range=(3.0, 8.0),
    dv_bins=(
        (-3.0, -1.0),   # ego slower
        (-0.5, 0.5),    # similar speed
        (1.0, 3.0),     # ego faster
    ),
):
   

    # ---------- 匝道位置分区 ----------
    s_bins_ramp = [
        (-90.0, -60.0),   # far
        (-60.0, -35.0),   # mid
        (-35.0, -10.0),   # near
    ]
    
    # ---------- 主路位置分区（ego已汇入主路）----------
    p_x_bins_main = [
        (0.0, 20.0),      # 刚汇入
        (20.0, 50.0),     # 汇入后一段距离
        (50.0, 100.0),    # 汇入后较远
    ]

    # ---------- 主路相对位置分区 ----------
    dx_bins = [
        (-80.0, -40.0),   # far behind
        (-40.0, -10.0),   # near behind
        (-10.0,  10.0),   # overlapping
        ( 10.0,  40.0),   # near ahead
        ( 40.0,  80.0),   # far ahead
    ]

    # 计算匝道采样和主路采样的数量分配
    num_bins_ramp = len(s_bins_ramp) * len(dx_bins) * len(dv_bins)
    num_bins_main = len(p_x_bins_main) * len(dx_bins) * len(dv_bins)
    num_bins_total = num_bins_ramp + num_bins_main
    
    N_per_bin = max(N // num_bins_total, 1)

    states = []

    def sample_on_ramp(n, s_min, s_max):
        s = get_stratified_tensor(n, s_min, s_max, device)
        p0 = p_merge.unsqueeze(0) + s.unsqueeze(1) * d_vec.unsqueeze(0)
        return p0[:, 0], p0[:, 1]
    
    def sample_on_main(n, p_x_min, p_x_max):
        """采样 ego 已汇入主路的位置"""
        p_x = get_stratified_tensor(n, p_x_min, p_x_max, device)
        p_y = torch.full((n,), y_merge.item(), device=device)
        return p_x, p_y

    # ---------- 联合采样：匝道上的 ego ----------
    for s_lo, s_hi in s_bins_ramp:
        for dx_lo, dx_hi in dx_bins:
            for dv_lo, dv_hi in dv_bins:

                n = N_per_bin

                # Ego position (在匝道上)
                p_x, p_y = sample_on_ramp(n, s_lo, s_hi)

                # Ego speed
                v1 = get_stratified_tensor(n, v1_range[0], v1_range[1], device)

                # Relative speed
                dv = get_stratified_tensor(n, dv_lo, dv_hi, device)
                v2 = torch.clamp(v1 - dv, 0.0, 10.0)

                # Relative longitudinal position
                dx = get_stratified_tensor(n, dx_lo, dx_hi, device)
                x2 = p_x + dx

                states.append(
                    torch.stack([v1, v2, p_x, p_y, x2], dim=1)
                )

    # ---------- 联合采样：已汇入主路的 ego ----------
    for p_x_lo, p_x_hi in p_x_bins_main:
        for dx_lo, dx_hi in dx_bins:
            for dv_lo, dv_hi in dv_bins:

                n = N_per_bin

                # Ego position (已汇入主路)
                p_x, p_y = sample_on_main(n, p_x_lo, p_x_hi)

                # Ego speed
                v1 = get_stratified_tensor(n, v1_range[0], v1_range[1], device)

                # Relative speed
                dv = get_stratified_tensor(n, dv_lo, dv_hi, device)
                v2 = torch.clamp(v1 - dv, 0.0, 10.0)

                # Relative longitudinal position
                dx = get_stratified_tensor(n, dx_lo, dx_hi, device)
                x2 = p_x + dx

                states.append(
                    torch.stack([v1, v2, p_x, p_y, x2], dim=1)
                )

    all_states = torch.cat(states, dim=0)

    # ---------- 截断 / 补齐 ----------
    if all_states.shape[0] > N:
        idx = torch.randperm(all_states.shape[0], device=device)[:N]
        all_states = all_states[idx]
    elif all_states.shape[0] < N:
        missing = N - all_states.shape[0]

        # 随机选择在匝道或主路补齐
        n_ramp = missing // 2
        n_main = missing - n_ramp
        
        if n_ramp > 0:
            p_x, p_y = sample_on_ramp(n_ramp, -90.0, -10.0)
            v1 = get_stratified_tensor(n_ramp, 3.0, 8.0, device)
            dv = get_stratified_tensor(n_ramp, -2.0, 2.0, device)
            v2 = torch.clamp(v1 - dv, 0.0, 10.0)
            dx = get_stratified_tensor(n_ramp, -80.0, 80.0, device)
            x2 = p_x + dx
            fill_ramp = torch.stack([v1, v2, p_x, p_y, x2], dim=1)
        else:
            fill_ramp = torch.empty((0, 5), device=device)
        
        if n_main > 0:
            p_x, p_y = sample_on_main(n_main, 0.0, 100.0)
            v1 = get_stratified_tensor(n_main, 3.0, 8.0, device)
            dv = get_stratified_tensor(n_main, -2.0, 2.0, device)
            v2 = torch.clamp(v1 - dv, 0.0, 10.0)
            dx = get_stratified_tensor(n_main, -80.0, 80.0, device)
            x2 = p_x + dx
            fill_main = torch.stack([v1, v2, p_x, p_y, x2], dim=1)
        else:
            fill_main = torch.empty((0, 5), device=device)
        
        all_states = torch.cat([all_states, fill_ramp, fill_main], dim=0)

    return all_states[torch.randperm(N, device=device)]

   





In [9]:
def get_mixed_initial_state_merge(N, device, mode: str = "default"):


    def sample_on_ramp(n, s_min, s_max):
        if n <= 0:
            return None
        s = get_stratified_tensor(n, s_min, s_max, device)  # [n]
        p0 = p_merge.unsqueeze(0) + s.unsqueeze(1) * d_vec.unsqueeze(0)
        p_x = p0[:, 0]
        p_y = p0[:, 1]
        return s, p_x, p_y
    
    if mode == "style0_gap":
        # Style 0 训练重点：虽然他凶，但要在夹缝中求生存
        # 减少必死场景 (Must Yield)，增加 "Rush Mid" (博弈)
        # ⭐ 提高 "wait_before_merge" 占比，让 Style 0 学会更谨慎地等待主路车通过后再 merge
        frac_edge = {
            "rush_mid": 0.10,      # 关键：双方速度差不多，看谁快（稍微降低）
            "close_fast": 0.08,    # 难：后车快
            "must_yield0": 0.05,   # 极难：必死 (从 30% 降到 5%！只要让它见过就行，别吓死它)
            "wait_before_merge": 0.22,  # ⭐ 提高：从 0.15 提高到 0.22，让 Style 0 更多学习等待策略
            "stop_near": 0.0,      # Style 0 不会停车
            "gapmerge_s1": 0.0
        }
    elif mode == "style2_gap":
        # Style 2 训练重点：识别让行（保守改进：适度增加采样）
        frac_edge = {
            "stop_near": 0.20,         # 静态让行（适度增加）
            "slow_yield": 0.20,        # 动态让行（适度增加）
            "ego_near_conflict": 0.10, # 新增：ego 接近冲突区
            "rush_mid": 0.05,
            "must_yield0": 0.0,
            "gapmerge_s1": 0.0
        }
    elif mode == "style1_gap":
        frac_edge = {
            "gapmerge_s1": 0.30,   # 专门练切入
            "rush_mid": 0.10,
            "close_fast": 0.05,    # 偶尔遇到不让的
            "stop_near": 0.0,
            "must_yield0": 0.0
        }
    else: # Default (混合)
        frac_edge = {
            "rush_mid": 0.10,
            "close_fast": 0.05,
            "stop_near": 0.05,
            "gapmerge_s1": 0.05,
            "must_yield0": 0.02
        }

    # 计算 Edge Cases 的总数
    N_edges = {}
    total_edge_N = 0
    for key, frac in frac_edge.items():
        n_count = int(N * frac)
        N_edges[key] = n_count
        total_edge_N += n_count

    N_rand = max(N - total_edge_N, 0)
    
    states_list = []
    if N_rand > 0:
        states_bb = sample_merge_joint_space(
            N=N_rand,
            device=device,
        )
        states_list.append(states_bb)

    # B) 注入 Edge Cases
    # 1. Rush-Mid (博弈)
    if N_edges.get("rush_mid", 0) > 0:
        n = N_edges["rush_mid"]
        _, p_x, p_y = sample_on_ramp(n, -60.0, -25.0)
        v1 = get_stratified_tensor(n, 4.0, 7.0, device)
        v2 = get_stratified_tensor(n, 4.0, 8.0, device)
        dx = get_stratified_tensor(n, -60.0, -20.0, device)
        x2 = p_x + dx
        states_list.append(torch.stack([v1, v2, p_x, p_y, x2], dim=1))

    # 2. Must-Yield-0 (必死/高难) - 即使是 Style0 模式也别给太多
    if N_edges.get("must_yield0", 0) > 0:
        n = N_edges["must_yield0"]
        _, p_x, p_y = sample_on_ramp(n, -40.0, -20.0) # 别太近，给一点反应时间
        v1 = get_stratified_tensor(n, 2.0, 5.0, device)
        v2 = get_stratified_tensor(n, 8.0, 12.0, device) # 后车很快
        dx = get_stratified_tensor(n, -20.0, -5.0, device)
        x2 = p_x + dx
        states_list.append(torch.stack([v1, v2, p_x, p_y, x2], dim=1))

    # ⭐ 新增：Wait-Before-Merge (Style 0 等待场景)
    # 让 Style 0 学会等待主路车通过后再 merge
    # 场景特征：主路车在前面且速度快，ego 需要等待
    if N_edges.get("wait_before_merge", 0) > 0:
        n = N_edges["wait_before_merge"]
        # ego 在匝道上，接近 merge 点（-50 到 -35），给足够的反应时间
        _, p_x, p_y = sample_on_ramp(n, -50.0, -35.0)
        # ego 速度中等（4-7 m/s），不会太快也不会太慢
        v1 = get_stratified_tensor(n, 4.0, 7.0, device)
        # 主路车速度较快（10-13 m/s），对应 Style 0 的激进配置
        v2 = get_stratified_tensor(n, 10.0, 13.0, device)
        # 主路车在前面（x2 > p_x），距离 ego 较近（5-25 m），让 ego 必须等待
        dx = get_stratified_tensor(n, 5.0, 25.0, device)  # 正数表示主路车在前面
        x2 = p_x + dx
        states_list.append(torch.stack([v1, v2, p_x, p_y, x2], dim=1))

    
    # Close-Fast (Style 1 常用)
    if N_edges.get("close_fast", 0) > 0:
        n = N_edges["close_fast"]
        _, p_x, p_y = sample_on_ramp(n, -30.0, -5.0)
        v1 = get_stratified_tensor(n, 4.0, 6.0, device)
        v2 = get_stratified_tensor(n, 6.0, 9.0, device)
        dx = get_stratified_tensor(n, -40.0, -10.0, device)
        x2 = p_x + dx
        states_list.append(torch.stack([v1, v2, p_x, p_y, x2], dim=1))
        
    # Gap-Merge (Style 1 专属)
    if N_edges.get("gapmerge_s1", 0) > 0:
        n = N_edges["gapmerge_s1"]
        _, p_x, p_y = sample_on_ramp(n, -35.0, -10.0)
        v1 = get_stratified_tensor(n, 4.0, 8.0, device)
        v2 = get_stratified_tensor(n, 4.0, 7.0, device)
        dx = get_stratified_tensor(n, -10.0, 20.0, device)
        x2 = p_x + dx
        states_list.append(torch.stack([v1, v2, p_x, p_y, x2], dim=1))

    # Stop-Near (Style 2 专属) - 改进：ego 更接近冲突区
    if N_edges.get("stop_near", 0) > 0:
        n = N_edges["stop_near"]
        x2 = get_stratified_tensor(n, STOP_X - 2.0, STOP_X + 2.0, device)
        v2 = get_stratified_tensor(n, 0.0, 1.0, device)
        _, p_x, p_y = sample_on_ramp(n, -35.0, -15.0)  # 改进：ego 从 [-50, -30] 改为 [-35, -15]
        v1 = get_stratified_tensor(n, 0.0, 5.0, device)
        states_list.append(torch.stack([v1, v2, p_x, p_y, x2], dim=1))

    # 场景：Ego 离汇入点近，后车虽然近但速度慢 -> 动态让行场景
    if N_edges.get("slow_yield", 0) > 0:
        n = N_edges["slow_yield"]
        _, p_x, p_y = sample_on_ramp(n, -30.0, -5.0)
        v1 = get_stratified_tensor(n, 4.0, 6.0, device)
        v2 = get_stratified_tensor(n, 2.0, 5.0, device)   # <--- 关键：后车很慢
        dx = get_stratified_tensor(n, -10.0, 20.0, device)
        x2 = p_x + dx
        states_list.append(torch.stack([v1, v2, p_x, p_y, x2], dim=1))

    # Ego-Near-Conflict (Style 2 关键场景)：ego 接近冲突区，主路车在 stop line 前
    if N_edges.get("ego_near_conflict", 0) > 0:
        n = N_edges["ego_near_conflict"]
        _, p_x, p_y = sample_on_ramp(n, -30.0, -20.0)  # ego 在 -30 到 -20，接近冲突区
        v1 = get_stratified_tensor(n, 3.0, 6.0, device)
        # 主路车在 stop line 前（-35 到 -26），正在减速
        x2 = get_stratified_tensor(n, -35.0, STOP_X - 2.0, device)
        v2 = get_stratified_tensor(n, 2.0, 5.0, device)  # 主路车速度较慢，正在减速
        states_list.append(torch.stack([v1, v2, p_x, p_y, x2], dim=1))


    # 拼接并打乱
    all_states = torch.cat(states_list, dim=0)
    current_len = all_states.shape[0]
    if current_len < N:
        missing = N - current_len
        _, p_x_r, p_y_r = sample_on_ramp(missing, -90.0, -10.0)
        v1_r = get_stratified_tensor(missing, 3.0, 8.0, device)
        v2_r = get_stratified_tensor(missing, 3.0, 9.0, device)
        dx_r = get_stratified_tensor(missing, -80.0, 80.0, device)
        x2_r = p_x_r + dx_r
        fill_states = torch.stack([v1_r, v2_r, p_x_r, p_y_r, x2_r], dim=1)
        all_states = torch.cat([all_states, fill_states], dim=0)

    if all_states.shape[0] > N:
        idx = torch.randperm(all_states.shape[0], device=device)[:N]
        all_states = all_states[idx]

    return all_states[torch.randperm(all_states.shape[0], device=device)]

In [10]:
states = get_mixed_initial_state_merge(5000, device)
dx = states[:, 4] - states[:, 2]
dv = states[:, 0] - states[:, 1]

print(
    "dx bins:",
    torch.histc(dx, bins=10, min=-80, max=80)
)
print(
    "dv bins:",
    torch.histc(dv, bins=10, min=-10, max=5)
)


dx bins: tensor([293., 443., 607., 699., 863., 745., 421., 343., 293., 293.])
dv bins: tensor([  10.,   27.,   42.,   41.,  432., 1222., 1564.,  858.,  755.,   49.])


In [ ]:
def _strat_mat(n: int, m: int, a: float, b: float, device, dtype):
# n 行，每行 m 个分层采样
    base = torch.linspace(a, b, steps=m, device=device, dtype=dtype) # [m]
    base = base.unsqueeze(0).expand(n, m) # [n,m]
    # 每个 cell 里加一点均匀扰动
    cell = (b - a) / m
    noise = torch.rand((n, m), device=device, dtype=dtype) * cell
    return base - cell + noise


def _make_gaps(n: int, K: int, gap_mode: str, device, dtype):
    if n <= 0:
        return torch.empty((0, K-1), device=device, dtype=dtype)

    if gap_mode == "tight":
        return _strat_mat(n, K-1, 5.0, 12.0, device, dtype)

    if gap_mode == "medium":
        return _strat_mat(n, K-1, 12.0, 20.0, device, dtype)

    if gap_mode == "loose":
        return _strat_mat(n, K-1, 20.0, 35.0, device, dtype)

    if gap_mode == "mix":
        g_t = _strat_mat(n, K-1, 5.0, 12.0, device, dtype)
        g_l = _strat_mat(n, K-1, 12.0, 20.0, device, dtype)
        mask = (torch.rand((n, K-1), device=device) < 0.5)
        return torch.where(mask, g_t, g_l)

    raise ValueError(f"Unknown gap_mode={gap_mode}")

def get_mixed_initial_state_merge_Ncars(
    N, K, device,
    gap_mode: str = "mix",
    base_mode: str = "default", # 新增：传给 2-cars 的 mode
    ensure_feasible: bool = True,
    min_gap_margin: float = 2.0,
    oversample: int = 4,
    max_tries: int = 50,

    ):
    dtype = torch.float32
    def _one_shot(B: int):
        # 1) 用 2-cars sampler 采 base state（注意这里传 base_mode）
        base_states = get_mixed_initial_state_merge(B, device, mode=base_mode) # [N, 5]
        v1_0 = base_states[:, 0].to(device=device, dtype=dtype)
        v2_0 = base_states[:, 1].to(device=device, dtype=dtype)
        ego_x0 = base_states[:, 2].to(device=device, dtype=dtype)
        ego_y0 = base_states[:, 3].to(device=device, dtype=dtype)
        x2_0 = base_states[:, 4].to(device=device, dtype=dtype)

        v1_0 = v1_0.to(device=device, dtype=dtype)
        v2_0 = v2_0.to(device=device, dtype=dtype)
        ego_x0 = ego_x0.to(device=device, dtype=dtype)
        ego_y0 = ego_y0.to(device=device, dtype=dtype)
        x2_0 = x2_0.to(device=device, dtype=dtype)

        gaps = _make_gaps(B, K, gap_mode=gap_mode, device=device, dtype=dtype) # [N, K-1]
        step = L + gaps
        zeros = torch.zeros((B, 1), device=device, dtype=dtype)
        offsets = -torch.cumsum(step, dim=1)
        offsets = torch.cat([zeros, offsets], dim=1)

        x_cars0 = x2_0.unsqueeze(1) + offsets
        v_cars0 = v2_0.unsqueeze(1).expand(B, K).clone()

        return ego_x0, ego_y0, v1_0, x_cars0, v_cars0

    if not ensure_feasible:
        return _one_shot(N)
    # ===== 过滤 + 补齐：直到凑够 N 个可行样本 =====
    xs, ys, vs, xcs, vcs = [], [], [], [], []
    collected = 0
    tries = 0

    while collected < N and tries < max_tries:
        tries += 1
        B = min((N - collected) * oversample, 8192)

        ego_x0, ego_y0, v1_0, x_cars0, v_cars0 = _one_shot(B)

        # ① t=0 碰撞过滤（用你已有的判定）
        coll0 = approx_collision_Ncars_torch(ego_x0, ego_y0, x_cars0)  # [B] bool

        # ② 最小净间距过滤（与你统计一致：min_dist - L）
        dx = ego_x0.view(-1, 1) - x_cars0                    # [B,K]
        dy = ego_y0.view(-1, 1) - float(y_merge)             # [B,1]
        dist = torch.sqrt(dx * dx + dy * dy)                 # [B,K]
        min_dist, _ = dist.min(dim=1)                        # [B]
        min_gap_net = min_dist - float(L)                    # [B]

        feasible = (~coll0) & (min_gap_net > float(min_gap_margin))
        idx = torch.nonzero(feasible, as_tuple=False).squeeze(1)

        if idx.numel() == 0:
            continue

        take = min(idx.numel(), N - collected)
        idx = idx[:take]

        xs.append(ego_x0[idx]); ys.append(ego_y0[idx]); vs.append(v1_0[idx])
        xcs.append(x_cars0[idx]); vcs.append(v_cars0[idx])
        collected += take

    if collected < N:
        # 如果你想严格一点，就 raise；如果想宽松点，可以返回 collected 的数量
        raise RuntimeError(
            f"Feasible sampling failed: collected {collected}/{N} "
            f"(min_gap_margin={min_gap_margin}, tries={tries})"
        )

    ego_x0 = torch.cat(xs, dim=0)
    ego_y0 = torch.cat(ys, dim=0)
    v1_0   = torch.cat(vs, dim=0)
    x_cars0 = torch.cat(xcs, dim=0)
    v_cars0 = torch.cat(vcs, dim=0)

    return ego_x0, ego_y0, v1_0, x_cars0, v_cars0
    


In [13]:
import torch.nn.functional as F

def build_state_from_phys(ego_x, ego_y, v1, x_cars, v_cars, style_id):
    """
    Inputs:
      ego_x, ego_y, v1: [N]
      x_cars, v_cars:   [N, K]
      style_id:
        - python int / scalar tensor: one style for the whole batch
        - tensor [N]               : one style per episode (broadcast to all K cars)
        - tensor [N, K]            : one style per vehicle
    Returns:
      state:     [N, 3 + 5K]
      style_ids: [N, K] long
    """
    N, K = x_cars.shape
    device = ego_x.device

    if torch.is_tensor(style_id):
        if style_id.dim() == 0:
            sid = int(style_id.item())
            style_ids = torch.full((N, K), sid, device=device, dtype=torch.long)
        elif style_id.dim() == 1:
            if style_id.shape[0] != N:
                raise ValueError(f"style_id has shape {tuple(style_id.shape)} but N={N}")
            style_ids = style_id.to(device=device, dtype=torch.long).view(N, 1).expand(N, K)
        elif style_id.dim() == 2:
            if style_id.shape != (N, K):
                raise ValueError(f"style_id has shape {tuple(style_id.shape)} but expected {(N, K)}")
            style_ids = style_id.to(device=device, dtype=torch.long)
        else:
            raise ValueError(f"style_id.dim()={style_id.dim()} is not supported")
    else:
        style_ids = torch.full((N, K), int(style_id), device=device, dtype=torch.long)

    style_onehot = F.one_hot(style_ids, num_classes=3).float()  # [N,K,3]
    style_flat   = style_onehot.view(N, 3 * K)                  # [N,3K]

    state = torch.cat(
        [
            v1.unsqueeze(1),
            ego_x.unsqueeze(1),
            ego_y.unsqueeze(1),
            x_cars,
            v_cars,
            style_flat,
        ],
        dim=1,
    )  # [N, 3 + 5K]

    return state, style_ids


In [14]:
class Critic(nn.Module):
    def __init__(self, K = K, T=T, hidden=256):
        super().__init__()
        self.K = K
        state_dim = 3 + 5 * K
        self.net = nn.Sequential(
            nn.Linear(state_dim + T, hidden), 
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1) # 输出预测的总 Cost
        )
        self.register_buffer('norm_v', torch.tensor(10.0))
        self.register_buffer('norm_p', torch.tensor(100.0))

    def forward(self, state, action_seq):
        K = self.K
        norm_s = state.clone()

        # 索引约定：
        #  0      : v1
        #  1      : p_x
        #  2      : p_y
        #  3..3+K-1         : x_cars
        #  3+K..3+2K-1      : v_cars
        #  其余 3K 维       : style one-hot（不归一化）

        v1_idx = 0
        px_idx = 1
        py_idx = 2
        x_start = 3
        x_end   = x_start + K
        v_start = x_end
        v_end   = v_start + K

        # 归一化 ego
        norm_s[:, v1_idx] = norm_s[:, v1_idx] / self.norm_v
        norm_s[:, px_idx] = norm_s[:, px_idx] / self.norm_p
        norm_s[:, py_idx] = norm_s[:, py_idx] / self.norm_p

        # 归一化所有 main road 车的位置、速度
        norm_s[:, x_start:x_end] = norm_s[:, x_start:x_end] / self.norm_p
        norm_s[:, v_start:v_end] = norm_s[:, v_start:v_end] / self.norm_v
        # style one-hot 部分保持原样

        # 展平 action_seq: [N,T] 或 [N,T,1] -> [N,T]
        if action_seq.dim() > 2:
            action_seq = action_seq.view(action_seq.size(0), -1)

        x = torch.cat([norm_s, action_seq], dim=1)  # [N, state_dim + T]
        return self.net(x)

# critic = Critic(K=K).to(device)
# optimizer_c = optim.Adam(critic.parameters(), lr= 1e-3)
# print(">>> 开始 Critic 预训练 (拟合物理公式)...")

In [15]:
class Actor(nn.Module):
    def __init__(self, K = K, hidden = 256): # 宽度增加到 256
        super().__init__()
        self.K = K
        state_dim = 3 + 5 * K
        self.net = nn.Sequential(
            # 第一层
            nn.Linear(state_dim, hidden), 
            nn.ReLU(),
            
            # 新增：第二层 (增加深度，提取更复杂的特征)
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            
            # 新增：第三层 (可选，为了更强的非线性能力)
            nn.Linear(hidden, hidden),
            nn.ReLU(),

            # 输出层
            nn.Linear(hidden, T) 
        )
        
        # 归一化参数保持不变
        self.register_buffer('norm_v', torch.tensor(10.0))
        self.register_buffer('norm_p', torch.tensor(100.0))

    def forward(self, s):
        K = self.K
        norm_s = s.clone()

        v1_idx = 0
        px_idx = 1
        py_idx = 2
        x_start = 3
        x_end   = x_start + K
        v_start = x_end
        v_end   = v_start + K

        norm_s[:, v1_idx] = norm_s[:, v1_idx] / self.norm_v
        norm_s[:, px_idx] = norm_s[:, px_idx] / self.norm_p
        norm_s[:, py_idx] = norm_s[:, py_idx] / self.norm_p
       
        norm_s[:, x_start:x_end] = norm_s[:, x_start:x_end] / self.norm_p
        norm_s[:, v_start:v_end] = norm_s[:, v_start:v_end] / self.norm_v
        
        raw = self.net(norm_s)
        
        # 输出限制在 [-u_max, u_max]
        u = torch.tanh(raw) * u_max
        return u
# actor = Actor(K = K).to(device)
# optimizer_actor = optim.Adam(actor.parameters(), lr=3e-4)


In [16]:
def sanity_check_critic_merge_style_Ncars(critic_net, style_id=1, REWARD_SCALE=600.0):
    """
    针对 N-cars / 新 state 的 Critic sanity check。
    仍然构造 4 个典型场景：
      0: Safe-Far
      1: Clear-Go
      2: Clear-Brake
      3: Crash-Gas
    检查预测排序是否满足：
      Safe-Far < Clear-Go < Crash-Gas < Clear-Brake
    """
    critic_net.eval()
    device = next(critic_net.parameters()).device

    with torch.no_grad():
        # ------- 构造 4 个场景的 ego 物理量 -------
        # 0) Safe-Far
        t_safe = torch.tensor(-40.0, device=device)
        p_safe = p_merge + d_vec * t_safe
        p_x_safe, p_y_safe = p_safe[0], p_safe[1]

        v1_safe = torch.tensor(5.0, device=device)   # 接近 v1_d
        v2_safe = torch.tensor(3.0, device=device)
        x2_safe = torch.tensor(80.0, device=device)  # 头车很远

        u_safe = torch.zeros(T, device=device)

        # 1/2) Clear 场景：ego 在 merge 口附近，car 队已经开走一段
        s_ca   = torch.tensor(-25.0, device=device)  # ego 刚好进入 -25
        p_ca   = p_merge + d_vec * s_ca
        p_x_ca, p_y_ca = p_ca[0], p_ca[1]

        v1_ca = torch.tensor(1.0, device=device)     # 低速蠕行
        v2_ca = torch.tensor(6.0, device=device)
        x2_ca = torch.tensor(-20.0, device=device)  # 主路车在前面，在 conflict zone 内

        u_clear_go    = torch.full((T,), 2.0, device=device)   # 略加速
        u_clear_brake = torch.full((T,), -u_max, device=device)  # 一路大刹车

        # 3) Crash-Gas：冲突区两车很近还猛踩油
        p_x_crash = torch.tensor(-5.0, device=device)
        p_y_crash = y_merge.clone()
        x2_crash  = torch.tensor(-6.0, device=device)

        v1_crash = torch.tensor(5.0, device=device)
        v2_crash = torch.tensor(5.0, device=device)
        u_crash  = torch.full((T,), u_max, device=device)

        # ------- 拼 batch：ego -------
        v1_batch  = torch.stack([v1_safe, v1_ca,  v1_ca,  v1_crash],  dim=0)
        ego_x_b   = torch.stack([p_x_safe, p_x_ca, p_x_ca, p_x_crash], dim=0)
        ego_y_b   = torch.stack([p_y_safe, p_y_ca, p_y_ca, p_y_crash], dim=0)

        # 头车 x / v
        x_head_b  = torch.stack([x2_safe, x2_ca, x2_ca, x2_crash], dim=0)
        v_head_b  = torch.stack([v2_safe, v2_ca, v2_ca, v2_crash], dim=0)

        N_scen = 4
        K_local = K  # 使用全局 K

        # ------- 根据头车构造 platoon：x_cars0, v_cars0 形状 [4, K] -------
        x_cars0 = torch.zeros((N_scen, K_local), device=device)
        v_cars0 = torch.zeros((N_scen, K_local), device=device)

        for i in range(N_scen):
            # 第 0 辆 = 头车
            x_cars0[i, 0] = x_head_b[i]
            # 所有车速度 = 头车速度
            v_cars0[i, :] = v_head_b[i]
            # 后面每辆往后排，间距 20m（+ 车长）
            for k in range(1, K_local):
                x_cars0[i, k] = x_cars0[i, k-1] - (L + 20.0)

        # style_ids: [4, K]，全是当前 style_id
        style_ids = torch.full(
            (N_scen, K_local), style_id,
            device=device, dtype=torch.long
        )

        # ------- 构造 state（用新版 build_state_from_phys） -------
        states, style_ids_out = build_state_from_phys(
            ego_x_b, ego_y_b, v1_batch,
            x_cars0, v_cars0,
            style_id  # 标量，内部会扩成 [4,K]
        )
        # style_ids_out: [4,K]，与 style_ids 一致，这里直接用即可
        style_ids = style_ids_out

        # 动作 batch
        actions = torch.stack(
            [u_safe, u_clear_go, u_clear_brake, u_crash],
            dim=0
        )  # [4, T]

        # ------- Critic 预测的 scaled cost -------
        pred = critic_net(states, actions).view(-1)  # [4]

        # ------- 物理真值 cost（scaled） -------
        true = potentialFunction_update_merge_Ncars(
            actions,
            ego_x_b, ego_y_b, v1_batch,
            x_cars0, v_cars0,
            style_ids,
            ego_v_d, alpha, beta, delta, T, dt,
        ).view(-1) / REWARD_SCALE

    labels = ["Safe-Far", "Clear-Go", "Clear-Brake", "Crash-Gas"]

    # 顺序检查：根据不同 Style 设置不同的期望排序
    # 根据不同 Style 设置不同的期望排序
    if style_id == 0:  # Style 0: Safe-Far < Clear-Go < Crash-Gas < Clear-Brake
        ok0 = pred[1] >= pred[0] - 1e-3  # Clear-Go ≥ Safe-Far
        ok1 = pred[3] >= pred[1] - 1e-3  # Crash-Gas ≥ Clear-Go
        ok2 = pred[2] >= pred[3] - 1e-3  # Clear-Brake ≥ Crash-Gas
    elif style_id == 1:  # Style 1: Safe-Far < Clear-Go < Crash-Gas < Clear-Brake
        ok0 = pred[1] >= pred[0] - 1e-3  # Clear-Go ≥ Safe-Far
        ok1 = pred[3] >= pred[1] - 1e-3  # Crash-Gas ≥ Clear-Go
        ok2 = pred[2] >= pred[3] - 1e-3  # Clear-Brake ≥ Crash-Gas
    else:  # Style 2: Safe-Far < Clear-Go < Crash-Gas < Clear-Brake
        ok0 = pred[1] >= pred[0] - 1e-3  # Clear-Go ≥ Safe-Far
        ok1 = pred[3] >= pred[1] - 1e-3  # Crash-Gas ≥ Clear-Go
        ok2 = pred[2] >= pred[3] - 1e-3  # Clear-Brake ≥ Crash-Gas

    all_ok = bool(ok0 & ok1 & ok2)

    if all_ok:
        print(f">>> ✅ Critic (style={style_id}) 在关键场景上的排序合理。")
    else:
        print(f">>> ❌ Critic (style={style_id}) 在关键场景上的排序不理想。")
        for i in range(4):
            print(f"{i}: {labels[i]:>12s} | pred={pred[i].item():8.3f} | true={true[i].item():8.3f}")

    return all_ok





In [34]:
r_ego = 2.5
r_car2 = 2.5
collision_dist2 = (r_ego + r_car2) ** 2
@torch.no_grad()
def approx_collision_Ncars_torch(p_x, p_y, x_cars):
    """
    兼容：
      - 单样本: p_x,p_y: 标量或 [1]；x_cars: [K]
      - batch : p_x,p_y: [B]；      x_cars: [B,K]
    永远返回 torch.bool tensor：
      - 单样本 -> shape [1]
      - batch  -> shape [B]
    """
    # 保证 tensor
    if not torch.is_tensor(p_x):
        p_x = torch.as_tensor(p_x, device=x_cars.device)
    if not torch.is_tensor(p_y):
        p_y = torch.as_tensor(p_y, device=x_cars.device)

    # 标量 -> [1]
    if p_x.dim() == 0:
        p_x = p_x.view(1)
    if p_y.dim() == 0:
        p_y = p_y.view(1)

    # x_cars: [K] -> [1,K]
    if x_cars.dim() == 1:
        x_cars = x_cars.unsqueeze(0)

    # 广播到 [B,K]
    dx = p_x.unsqueeze(1) - x_cars          # [B,K]
    dy = p_y.unsqueeze(1) - y_merge         # [B,1] -> broadcast
    dist2 = dx * dx + dy * dy               # [B,K]
    return (dist2 <= collision_dist2).any(dim=1)  # [B] bool tensor




In [32]:
def evaluate_policy_fixed_states_merge_Ncars(
    actor_net,
    init_npz_path: str,
    num_episodes_per_style: int = None,
    max_steps: int = 60,
    K_local=None,
    device=None,
    success_x: float = 50.0,
    # ===== 新增：失败导出 =====
    dump_fail_npz: str = None,          # e.g., "fails_epoch1000.npz"
    dump_max_per_style: int = 999999,   # 每个 style 最多存多少条失败
    dump_store_traj: bool = False,      # 是否存轨迹（默认不存，先存初始就够）
):
    if device is None:
        device = next(actor_net.parameters()).device
    if K_local is None:
        K_local = K

    data = np.load(init_npz_path, allow_pickle=False)

    style_list_np = data["style_list"].astype(np.int64)
    style_list = [int(x) for x in style_list_np.tolist()]
    M = len(style_list)

    ego_x0_all = torch.from_numpy(data["ego_x0"]).to(device).float()      # [M,E]
    ego_y0_all = torch.from_numpy(data["ego_y0"]).to(device).float()
    v1_0_all   = torch.from_numpy(data["v1_0"]).to(device).float()
    x_cars_all = torch.from_numpy(data["x_cars0"]).to(device).float()     # [M,E,K]
    v_cars_all = torch.from_numpy(data["v_cars0"]).to(device).float()

    E_total = ego_x0_all.shape[1]
    E = E_total if (num_episodes_per_style is None) else min(num_episodes_per_style, E_total)

    actor_net.eval()

    results = {
        "per_style": {},
        "overall": {"success": 0, "collision": 0, "timeout": 0, "avg_speed": 0.0},
        "meta": {"M": M, "E": E, "max_steps": max_steps, "K": K_local, "init_npz_path": init_npz_path},
    }

    # ===== 新增：失败缓存（先用 python list 收集，最后一次性转 numpy）=====
    fail_style   = []
    fail_ep      = []
    fail_type    = []   # 1=collision, 2=timeout
    fail_step    = []   # 发生失败的 step（collision 才有意义，timeout 是 max_steps-1）
    fail_ego_x0  = []
    fail_ego_y0  = []
    fail_v1_0    = []
    fail_x_cars0 = []
    fail_v_cars0 = []
    fail_s0      = []   # 额外存一个对你很有用的特征：初始 s(t) 距 merge 的 signed_t

    # 可选轨迹
    traj_px = []
    traj_v1 = []
    traj_min_dist2 = []

    overall_v_sum = 0.0
    overall_cnt = 0

    with torch.no_grad():
        for mi, sid in enumerate(style_list):
            passed_cnt = 0
            collision_cnt = 0
            timeout_cnt = 0
            v_avg_sum = 0.0

            # 每个 style 控制最多存多少条失败
            stored_fail_this_style = 0

            for ep in range(E):
                # ---------- 取固定初始状态 ----------
                v1   = v1_0_all[mi, ep].clone()
                p_x  = ego_x0_all[mi, ep].clone()
                p_y  = ego_y0_all[mi, ep].clone()

                x_cs0 = x_cars_all[mi, ep, :K_local].clone()  # [K]
                v_cs0 = v_cars_all[mi, ep, :K_local].clone()  # [K]
                x_cs = x_cs0.clone()
                v_cs = v_cs0.clone()

                style_ids_b = torch.full((1, K_local), sid, device=device, dtype=torch.long)

                ep_v_sum = 0.0
                collision = False
                success_ep = False

                # 可选轨迹记录（只记录失败 episode；先暂存到本地 list）
                _px_list = []
                _v1_list = []
                _md_list = []

                for step in range(max_steps):
                    ego_x_b = p_x.view(1)
                    ego_y_b = p_y.view(1)
                    v1_b    = v1.view(1)
                    x_b     = x_cs.view(1, K_local)
                    v_b     = v_cs.view(1, K_local)

                    state_b, _ = build_state_from_phys(
                        ego_x_b, ego_y_b, v1_b,
                        x_b, v_b, style_ids_b
                    )

                    u_seq = actor_net(state_b)      # [1,T]
                    u1    = u_seq[0, 0]

                    # ===== Ego dynamics + geometry update（保持与你 cost 一致）=====
                    v1_b = v1_b + u1.view(1) * dt
                    step_len = v1_b * dt

                    s_old   = signed_t_from_merge_torch(ego_x_b, ego_y_b)
                    on_ramp = s_old < 0.0
                    cross_merge = on_ramp & (step_len > -s_old)

                    move_ramp = on_ramp & (~cross_merge)
                    ego_x_b = torch.where(move_ramp, ego_x_b + d_vec[0] * step_len, ego_x_b)
                    ego_y_b = torch.where(move_ramp, ego_y_b + d_vec[1] * step_len, ego_y_b)

                    remain = step_len + s_old
                    ego_x_b = torch.where(cross_merge, ego_x_b + d_vec[0] * (-s_old), ego_x_b)
                    ego_y_b = torch.where(cross_merge, ego_y_b + d_vec[1] * (-s_old), ego_y_b)
                    ego_x_b = torch.where(cross_merge, ego_x_b + remain, ego_x_b)

                    on_main = ~on_ramp
                    ego_x_b = torch.where(on_main, ego_x_b + step_len, ego_x_b)

                    v1 = v1_b[0]
                    p_x = ego_x_b[0]
                    p_y = ego_y_b[0]

                    # ===== ego_on_main（与你环境一致）=====
                    ego_front_y = ego_y_b + (L / 2.0) * d_vec[1]
                    MERGE_Y_FRONT_THRESHOLD = -14.0
                    ego_on_main = (ego_front_y > MERGE_Y_FRONT_THRESHOLD)

                    leader_x, leader_v, has_leader = compute_leaders_torch(
                        x_b, v_b, style_ids_b,
                        ego_x_b, v1_b,
                        ego_on_main,
                        lookahead=60,
                    )

                    a_cars_b = step_vehicle_Ncars_torch(
                        x_b, v_b,
                        leader_x, leader_v, has_leader,
                        ego_x_b, ego_y_b,
                        v1_b, u1.view(1), ego_v_d,
                        style_ids_b
                    )

                    v_cs = torch.clamp(v_cs + a_cars_b[0] * dt, min=0.0)
                    x_cs = x_cs + v_cs * dt

                    ep_v_sum += float(v1.item())

                    # 可选轨迹统计（最小距离）
                    if dump_store_traj:
                        dx = p_x - x_cs
                        dy = p_y - y_merge
                        dist2 = dx * dx + dy * dy
                        _px_list.append(float(p_x.item()))
                        _v1_list.append(float(v1.item()))
                        _md_list.append(float(dist2.min().item()))

                    # ===== 终止条件 =====
                    if approx_collision_Ncars_torch(p_x, p_y, x_cs).any().item():
                        collision = True
                        break
                    if float(p_x.item()) > success_x:
                        success_ep = True
                        break

                # episode 收尾
                if collision:
                    collision_cnt += 1
                elif success_ep:
                    passed_cnt += 1
                else:
                    timeout_cnt += 1

                v_avg = ep_v_sum / float(step + 1)
                v_avg_sum += v_avg
                overall_v_sum += v_avg
                overall_cnt += 1

                # ===== 新增：把失败 episode 写入 fail buffer =====
                if (not success_ep) and (stored_fail_this_style < dump_max_per_style):
                    stored_fail_this_style += 1

                    fail_style.append(int(sid))
                    fail_ep.append(int(ep))
                    fail_step.append(int(step))
                    fail_type.append(1 if collision else 2)  # 1=collision, 2=timeout

                    # 初始状态（注意：必须用“初始”，不要用 rollout 后的 p_x/x_cs）
                    fail_ego_x0.append(float(ego_x0_all[mi, ep].item()))
                    fail_ego_y0.append(float(ego_y0_all[mi, ep].item()))
                    fail_v1_0.append(float(v1_0_all[mi, ep].item()))
                    fail_x_cars0.append(x_cs0.detach().cpu().numpy().astype(np.float32))
                    fail_v_cars0.append(v_cs0.detach().cpu().numpy().astype(np.float32))

                    s0 = signed_t_from_merge_torch(
                        ego_x0_all[mi, ep].view(1),
                        ego_y0_all[mi, ep].view(1),
                    )[0]
                    fail_s0.append(float(s0.item()))

                    if dump_store_traj:
                        traj_px.append(np.array(_px_list, dtype=np.float32))
                        traj_v1.append(np.array(_v1_list, dtype=np.float32))
                        traj_min_dist2.append(np.array(_md_list, dtype=np.float32))

            results["per_style"][int(sid)] = {
                "success_rate": passed_cnt / E,
                "collision_rate": collision_cnt / E,
                "timeout_rate": timeout_cnt / E,
                "avg_speed": v_avg_sum / E,
                "counts": {"success": passed_cnt, "collision": collision_cnt, "timeout": timeout_cnt, "E": E},
            }

            results["overall"]["success"]   += passed_cnt
            results["overall"]["collision"] += collision_cnt
            results["overall"]["timeout"]   += timeout_cnt

    total_eps = M * E
    results["overall"]["success_rate"]   = results["overall"]["success"] / total_eps
    results["overall"]["collision_rate"] = results["overall"]["collision"] / total_eps
    results["overall"]["timeout_rate"]   = results["overall"]["timeout"] / total_eps
    results["overall"]["avg_speed"]      = overall_v_sum / max(1, overall_cnt)

    # ===== 新增：保存失败 npz =====
    if dump_fail_npz is not None:
        # 变长轨迹不能直接堆成矩阵，用 object 保存（只有 dump_store_traj=True 才会用）
        save_dict = dict(
            style=np.array(fail_style, dtype=np.int64),
            ep=np.array(fail_ep, dtype=np.int64),
            fail_type=np.array(fail_type, dtype=np.int64),
            fail_step=np.array(fail_step, dtype=np.int64),
            ego_x0=np.array(fail_ego_x0, dtype=np.float32),
            ego_y0=np.array(fail_ego_y0, dtype=np.float32),
            v1_0=np.array(fail_v1_0, dtype=np.float32),
            s0=np.array(fail_s0, dtype=np.float32),
            x_cars0=np.stack(fail_x_cars0, axis=0) if len(fail_x_cars0) > 0 else np.zeros((0, K_local), np.float32),
            v_cars0=np.stack(fail_v_cars0, axis=0) if len(fail_v_cars0) > 0 else np.zeros((0, K_local), np.float32),
            meta=np.array([init_npz_path], dtype=object),
        )

        if dump_store_traj:
            save_dict["traj_px"] = np.array(traj_px, dtype=object)
            save_dict["traj_v1"] = np.array(traj_v1, dtype=object)
            save_dict["traj_min_dist2"] = np.array(traj_min_dist2, dtype=object)

        np.savez_compressed(dump_fail_npz, **save_dict)
        print(f"[Saved fail dump] -> {dump_fail_npz} | num_fails={len(fail_style)}")

        # 顺便给一个很实用的快速统计
        if len(fail_style) > 0:
            st = np.array(fail_style)
            ft = np.array(fail_type)
            for sid in sorted(set(st.tolist())):
                m = (st == sid)
                n = int(m.sum())
                n_col = int(((ft == 1) & m).sum())
                n_to  = int(((ft == 2) & m).sum())
                print(f"  style {sid}: fails={n} (collision={n_col}, timeout={n_to})")

    actor_net.train()
    return results


In [ ]:
FIXED_NPZ = r"c:\Users\zihanx\Desktop\review\AC_car\merge_code\merge_final_version\fixed_init_states.npz"
def make_fixed_init_states_merge_Ncars(
    save_path=FIXED_NPZ,
    num_episodes_per_style=200,
    style_list=(0, 1, 2),
    K_local=3,
    device=None,
    seed=0,
    # ===== 新增：可行性过滤参数 =====
    min_gap_margin=2.0,   # 增大安全余量，防止采样到"刚好不碰撞"的边界状态
    oversample=4,         # 每轮多采样倍数（避免 while 太慢）
):
    import numpy as np
    import torch

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    torch.manual_seed(seed)
    np.random.seed(seed)

    M = len(style_list)
    E = num_episodes_per_style
    K = K_local

    ego_x0 = torch.zeros((M, E), device=device)
    ego_y0 = torch.zeros((M, E), device=device)
    v1_0   = torch.zeros((M, E), device=device)
    x_cars0 = torch.zeros((M, E, K), device=device)
    v_cars0 = torch.zeros((M, E, K), device=device)

    # 你环境里用到的常量（确保这里能访问）
    # y_merge, L, get_style_config, get_mixed_initial_state_merge_Ncars, approx_collision_Ncars_torch
    # 如果这些在外部是全局变量，直接用即可。

    with torch.no_grad():
        for mi, sid in enumerate(style_list):
            collected = 0
            tries = 0
            while collected < E:
                tries += 1
                B = min((E - collected) * oversample, 4096)

                gap_mode, base_mode = get_style_config(sid)
                ex, ey, ev, xc, vc = get_mixed_initial_state_merge_Ncars(
                    B, K, device, gap_mode=gap_mode, base_mode=base_mode,
                    ensure_feasible=True,
                    min_gap_margin= min_gap_margin,
                    oversample=oversample            
                )  # ex,ey,ev: [B], xc,vc: [B,K]

                # 1) t=0 碰撞过滤
                coll0 = approx_collision_Ncars_torch(ex, ey, xc)  # [B] bool

                # 2) 最小净间距过滤（用你 CSV 里同样的定义：min_dist - L）
                dx = ex.view(-1, 1) - xc                    # [B,K]
                dy = ey.view(-1, 1) - float(y_merge)        # [B,1] broadcast
                dist = torch.sqrt(dx * dx + dy * dy)        # [B,K]
                min_dist, _ = dist.min(dim=1)               # [B]
                min_gap_net = min_dist - float(L)           # [B]

                feasible = (~coll0) & (min_gap_net > float(min_gap_margin))

                idx = torch.nonzero(feasible, as_tuple=False).squeeze(1)
                if idx.numel() == 0:
                    # 这一轮没采到可行的，继续采
                    if tries % 10 == 0:
                        print(f"[fixed_init] style {sid}: still collecting... ({collected}/{E}), tries={tries}")
                    continue

                take = min(idx.numel(), E - collected)
                idx = idx[:take]

                ego_x0[mi, collected:collected+take] = ex[idx]
                ego_y0[mi, collected:collected+take] = ey[idx]
                v1_0[mi, collected:collected+take]   = ev[idx]
                x_cars0[mi, collected:collected+take, :] = xc[idx]
                v_cars0[mi, collected:collected+take, :] = vc[idx]

                collected += take

            print(f"[fixed_init] style {sid}: collected {E}/{E} feasible inits (min_gap_margin={min_gap_margin})")

    np.savez(
        save_path,
        style_list=np.array(list(style_list), dtype=np.int64),
        ego_x0=ego_x0.cpu().numpy().astype(np.float32),
        ego_y0=ego_y0.cpu().numpy().astype(np.float32),
        v1_0=v1_0.cpu().numpy().astype(np.float32),
        x_cars0=x_cars0.cpu().numpy().astype(np.float32),
        v_cars0=v_cars0.cpu().numpy().astype(np.float32),
    )
    print(f"[fixed_init] saved -> {save_path}")


In [45]:
import os
import torch
import torch.optim as optim

SAVE_DIR  = r"c:\Users\zihanx\Desktop\review\AC_car"
best_path = os.path.join(SAVE_DIR, "AC_merge_actor_withstyle_Ncar314.pth")
best_critic_path = os.path.join(SAVE_DIR, "AC_merge_critic_withstyle_Ncar314.pth")

# 1. 重新实例化网络（确保结构一致）
actor = Actor(K=K).to(device)
critic = Critic(K=K).to(device)

# 2. 加载 best model
if os.path.isfile(best_path):
    actor.load_state_dict(torch.load(best_path, map_location=device))
    print(f">>> ✅ Loaded BEST actor from {best_path}")
else:
    print(f">>> ❌ WARN: best actor path not found: {best_path}")

if os.path.isfile(best_critic_path):
    critic.load_state_dict(torch.load(best_critic_path, map_location=device))
    print(f">>> ✅ Loaded BEST critic from {best_critic_path}")
else:
    print(f">>> ❌ WARN: best critic path not found: {best_critic_path}")

# 3. 重新初始化优化器（使用较低的学习率，因为从已训练的模型继续）
critic_lr = 1e-4  # 从 best model 继续，用较低学习率
actor_lr = 1e-4

optimizer_c = optim.Adam(critic.parameters(), lr=critic_lr)
optimizer_actor = optim.Adam(actor.parameters(), lr=actor_lr)

print(f">>> Optimizers initialized with LR: Critic={critic_lr:.2e}, Actor={actor_lr:.2e}")


>>> ✅ Loaded BEST actor from c:\Users\zihanx\Desktop\review\AC_car\AC_merge_actor_withstyle_Ncar314.pth
>>> ✅ Loaded BEST critic from c:\Users\zihanx\Desktop\review\AC_car\AC_merge_critic_withstyle_Ncar314.pth
>>> Optimizers initialized with LR: Critic=1.00e-04, Actor=1.00e-04


In [46]:

best_path = os.path.join(SAVE_DIR, "AC_merge_actor_withstyle_N.pth")
best_critic_path = os.path.join(SAVE_DIR, "AC_merge_critic_withstyle_N.pth")

In [ ]:
num_epochs   = 800
N_per_style  = 8000
# style_list   = [0, 1, 2]    
# M_styles     = len(style_list)
REWARD_SCALE = 600.0
ego_v_d = 5.0
for epoch in range(num_epochs):
    total_critic_loss = torch.tensor(0.0, device=device)
    # --- sample N initial states for this style ---
    for style_id in style_list:
        if style_id == STYLE_AGGRESSIVE:
            gap_mode  = "tight"
            base_mode = "style0_gap"
        elif style_id == STYLE_REACTIVE:
            gap_mode  = "loose"
            base_mode = "style1_gap"   # 这里启用专门的 gap-merge 分布
        else:  # STYLE_YIELD
            gap_mode  = "medium"
            base_mode = "style2_gap"
        ego_x0, ego_y0, v1_0, x_cars0, v_cars0 = \
            get_mixed_initial_state_merge_Ncars(
                    N_per_style, K, device,
                    gap_mode=gap_mode,
                    base_mode=base_mode,
                    ensure_feasible=True,
                    min_gap_margin= min_gap_margin,
                    oversample=oversample
                )
                
        state0, style_ids = build_state_from_phys(
            ego_x0, ego_y0, v1_0,
            x_cars0, v_cars0, style_id
        )
        
        random_actions = torch.empty(N_per_style, T, device=device).uniform_(u_min, u_max)

        with torch.no_grad():
            J_tot = potentialFunction_update_merge_Ncars(
                            random_actions, ego_x0, ego_y0, v1_0, x_cars0, v_cars0,
                            style_ids, ego_v_d, alpha, beta, delta, T, dt)              
            
            target_cost = (J_tot / REWARD_SCALE).unsqueeze(1)
        # ====== 4) critic loss for this style ======
        #flat_actions   = u_ego_seq.detach().reshape(N_per_style, -1)   
        predicted_values  = critic(state0, random_actions)         
        critic_loss_j  = ((predicted_values - target_cost) ** 2).mean()    

        total_critic_loss = total_critic_loss + critic_loss_j

    # ====== 5) update critic: average over styles ======
    total_critic_loss = total_critic_loss / M_styles

    optimizer_c.zero_grad()
    total_critic_loss.backward()
    torch.nn.utils.clip_grad_norm_(critic.parameters(), 1.0)
    optimizer_c.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Critic Loss: {total_critic_loss.item():.4f}")


Epoch 0, Critic Loss: 168.1679
Epoch 100, Critic Loss: 4.6784
Epoch 200, Critic Loss: 4.7833
Epoch 300, Critic Loss: 3.7971
Epoch 400, Critic Loss: 4.1305
Epoch 500, Critic Loss: 4.4076
Epoch 600, Critic Loss: 4.2201
Epoch 700, Critic Loss: 4.4470


In [48]:
for sid in style_list:
    sanity_ok = sanity_check_critic_merge_style_Ncars(
        critic,
        style_id=sid,
        REWARD_SCALE=REWARD_SCALE,
    )

>>> ✅ Critic (style=0) 在关键场景上的排序合理。
>>> ✅ Critic (style=1) 在关键场景上的排序合理。
>>> ✅ Critic (style=2) 在关键场景上的排序合理。


In [49]:
# =========================================================
# 辅助函数：提取重复的 style_id 映射
# =========================================================
def get_style_config(style_id):
    """根据 style_id 返回对应的 gap_mode 和 base_mode"""
    if style_id == STYLE_AGGRESSIVE:
        return "tight", "style0_gap"
    elif style_id == STYLE_REACTIVE:
        return "loose", "style1_gap"
    else:  # STYLE_YIELD
        return "medium", "style2_gap"

print(">>> Helper function 'get_style_config' defined")

>>> Helper function 'get_style_config' defined


In [50]:

def get_param_vector(model: torch.nn.Module) -> torch.Tensor:
    with torch.no_grad():
        return torch.cat([p.view(-1) for p in model.parameters()])

param_hist = []
prev_param = get_param_vector(actor)



In [51]:
delta_hist = []
theta_list = []  
loss_hist = []

In [ ]:
# =========================================================
# 改进的训练循环（带学习率调度和详细日志）
# =========================================================
import numpy as np

# 训练参数配置
num_epochs   = 400
N_per_style  = 4096
REWARD_SCALE = 600.0
#0.2
sigma0       = 0.4 * u_max
sigma_min    = 0.01 * u_max
decay_epochs = 800
critic_lr_start = 1e-4       # Critic 可以学快点
actor_lr_start  = 3e-4       # Actor 慢一点，防止崩塌
# Critic 和 Actor 更新比例
critic_update_steps = 3  # critic 每次更新 5 次
actor_update_steps = 1   # actor 每次更新 1 次

# 学习率调度器（可选：如果不想用调度器，可以注释掉）#13-5
try:
    critic_lr_schedule = optim.lr_scheduler.CosineAnnealingLR(
        optimizer_c, T_max=num_epochs, eta_min=1e-6
    )
    actor_lr_schedule = optim.lr_scheduler.CosineAnnealingLR(
        optimizer_actor, T_max=num_epochs, eta_min=1e-6
    )
    use_lr_schedule = True
    print(">>> Learning rate schedulers initialized")
except:
    use_lr_schedule = False
    print(">>> Warning: Could not initialize LR schedulers, using fixed LR")

# 训练指标记录
critic_loss_hist = []
actor_loss_hist = []

print(f">>> Starting training: {num_epochs} epochs, Critic steps: {critic_update_steps}, Actor steps: {actor_update_steps}")

for epoch in range(num_epochs):

    # =========================================================
    # 1) Critic phase: 多步更新 critic
    # =========================================================
    epoch_critic_losses = []
    
    for critic_step in range(critic_update_steps):
        total_critic_loss = torch.tensor(0.0, device=device)

        for style_id in style_list:
            gap_mode, base_mode = get_style_config(style_id)
            
            # 采样批次
            ego_x0, ego_y0, v1_0, x_cars0, v_cars0 = \
                get_mixed_initial_state_merge_Ncars(
                        N_per_style, K, device,
                        gap_mode=gap_mode,
                        base_mode=base_mode,
                        ensure_feasible=True,
                        min_gap_margin=2.0,
                        oversample=4
                    )
        
            state0, style_ids = build_state_from_phys(
                ego_x0, ego_y0, v1_0,
                x_cars0, v_cars0, style_id
            )

            # 计算噪声系数
            frac  = max(0.0, 1.0 - epoch / decay_epochs)
            sigma = sigma_min + (sigma0 - sigma_min) * frac

            with torch.no_grad():
                actions_mean = actor(state0)
                if sigma > 0:
                    noise   = torch.randn_like(actions_mean) * sigma
                    a_noisy = torch.clamp(actions_mean + noise, u_min, u_max)
                else:
                    a_noisy = actions_mean
                mix_p = 0.7 * frac + 0.2
                mask = (torch.rand((actions_mean.shape[0], 1), device=actions_mean.device) < mix_p)
                a_train = torch.where(mask, a_noisy, actions_mean)
                # 计算目标值
                J_tot = potentialFunction_update_merge_Ncars(
                                a_train, ego_x0, ego_y0, v1_0, x_cars0, v_cars0,
                                style_ids, ego_v_d, alpha, beta, delta, T, dt)
                J_target_z = (J_tot / REWARD_SCALE).unsqueeze(1)

            # Critic loss
            Q_pred = critic(state0, a_train)
            L_critic_z = ((Q_pred - J_target_z) ** 2).mean()
            total_critic_loss = total_critic_loss + L_critic_z

        # 对所有 style 取平均
        total_critic_loss = total_critic_loss / M_styles

        # Critic 更新
        optimizer_c.zero_grad()
        total_critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(critic.parameters(), 1.0)
        optimizer_c.step()
        
        epoch_critic_losses.append(total_critic_loss.item())

    # 记录 critic loss（取平均）
    avg_critic_loss = np.mean(epoch_critic_losses)
    critic_loss_hist.append(avg_critic_loss)

    # =========================================================
    # 2) Actor phase: 更新 actor
    # =========================================================
    total_actor_loss = torch.tensor(0.0, device=device)
    q_stats = {}

    for style_id in style_list:
        gap_mode, base_mode = get_style_config(style_id)
        
        # 重新采样批次
        ego_x0, ego_y0, v1_0, x_cars0, v_cars0 = \
            get_mixed_initial_state_merge_Ncars(
                    N_per_style, K, device,
                    gap_mode=gap_mode,
                    base_mode=base_mode,
                    ensure_feasible=True,
                    min_gap_margin=2.0,
                    oversample=4
                )
        
        state0_original, style_ids = build_state_from_phys(
            ego_x0, ego_y0, v1_0,
            x_cars0, v_cars0, style_id
        )

        # Actor loss
        actions_for_update = actor(state0_original)
        Q_est = critic(state0_original, actions_for_update)
        L_actor_z = Q_est.mean()

        total_actor_loss = total_actor_loss + L_actor_z
        # -------- 统计（不影响反传）--------
        with torch.no_grad():
            q_mean = float(Q_est.mean().item())
            q_std  = float(Q_est.std(unbiased=False).item())
            a_mean = float(actions_for_update.mean().item())
            a_std  = float(actions_for_update.std(unbiased=False).item())

        q_stats[int(style_id)] = {
            "q_mean": q_mean,
            "q_std":  q_std,
            "a_mean": a_mean,
            "a_std":  a_std,
        }
    total_actor_loss = total_actor_loss / M_styles

    # Actor 更新
    optimizer_actor.zero_grad()
    total_actor_loss.backward()
    torch.nn.utils.clip_grad_norm_(actor.parameters(), 1.0)
    optimizer_actor.step()
    
    # 记录指标
    actor_loss_hist.append(total_actor_loss.item())
    loss_hist.append(total_actor_loss.item())
    curr_param = get_param_vector(actor).detach().cpu().clone()
    theta_list.append(curr_param)

    # 更新学习率
    if use_lr_schedule:
        critic_lr_schedule.step()
        actor_lr_schedule.step()

    # =========================================================
    # 3) 评估 + 保存 best model
    # =========================================================
    if epoch % 100 == 0:
        print("  [Actor-Q stats] per style:")
        for sid in style_list:
            s = q_stats[int(sid)]
            print(
                f"    style {int(sid)}: "
                f"Q_mean={s['q_mean']:.4f}, Q_std={s['q_std']:.4f}, "
                f"a_mean={s['a_mean']:.4f}, a_std={s['a_std']:.4f}"
            )
    if epoch % 100 == 0:
        current_critic_lr = optimizer_c.param_groups[0]['lr']
        current_actor_lr  = optimizer_actor.param_groups[0]['lr']

        print(
            f"\n[Epoch {epoch}] "
            f"Critic Loss: {avg_critic_loss:.4f}, "
            f"Actor Loss: {total_actor_loss.item():.4f}, "
            f"Critic LR: {current_critic_lr:.2e}, "
            f"Actor LR: {current_actor_lr:.2e}, "
            f"Sigma: {sigma:.4f}"
        )

        # 只需要调用一次：它会评估所有 style
        res = evaluate_policy_fixed_states_merge_Ncars(
        actor_net=actor,
        init_npz_path="fixed_init_states.npz",
        num_episodes_per_style=200,
        max_steps=100,
        K_local=K,
        device=device,
        success_x=50.0,
        dump_fail_npz=rf"c:\Users\zihanx\Desktop\review\AC_car\merge_code\merge_final_version\fails_epoch{epoch}.npz",  
        dump_max_per_style=200,                      # 每个 style 最多保存 200 条失败
        dump_store_traj=False,                       # 先别存轨迹，初始态足够定位失败簇
    )


        # -------- 打印每个 style 的指标 --------
        success_total = 0.0
        speed_total   = 0.0
        coll_total    = 0.0

        for sid in style_list:
            m = res["per_style"][int(sid)]
            succ = m["success_rate"]
            coll = m["collision_rate"]
            spd  = m["avg_speed"]

            print(f"  [Style {sid}] Success={succ*100:.1f}%, Coll={coll*100:.1f}%, Speed={spd:.2f}")

            success_total += succ
            speed_total   += spd
            coll_total    += coll

        n_styles = len(style_list)
        success_mean = success_total / n_styles
        speed_mean   = speed_total   / n_styles
        coll_mean    = coll_total    / n_styles

        # overall（可选：你也可以直接用 res["overall"] 的 success_rate 等）
        overall = res["overall"]
        print(
            f"  [Overall] Success={overall['success_rate']*100:.1f}%, "
            f"Coll={overall['collision_rate']*100:.1f}%, "
            f"Speed={overall['avg_speed']:.2f}"
        )

        # -------- 你原来的 score / best 保存逻辑 --------
        current_score = success_mean * 10.0 - abs(speed_mean - float(ego_v_d))

        if current_score > best_score and success_mean > 0.85:
            best_score = current_score
            torch.save(actor.state_dict(), best_path)
            torch.save(critic.state_dict(), best_critic_path)
            print(f">>> Saved BEST model (score={current_score:.2f}) to: {best_path}")

        # sanity check（保持你原逻辑）
        if epoch % 500 == 0 and epoch > 0:
            print("\n>>> Checking Critic Sanity...")
            for sid in style_list:
                sanity_ok = sanity_check_critic_merge_style_Ncars(
                    critic,
                    style_id=sid,
                    REWARD_SCALE=REWARD_SCALE,
                )


>>> Learning rate schedulers initialized
>>> Starting training: 400 epochs, Critic steps: 3, Actor steps: 1
  [Actor-Q stats] per style:
    style 0: Q_mean=0.0289, Q_std=0.1787, a_mean=-0.1089, a_std=4.1312
    style 1: Q_mean=0.0402, Q_std=0.2822, a_mean=0.0190, a_std=3.9905
    style 2: Q_mean=0.1040, Q_std=0.2496, a_mean=-0.0617, a_std=3.9903

[Epoch 0] Critic Loss: 1.8089, Actor Loss: 0.0577, Critic LR: 1.00e-04, Actor LR: 1.00e-04, Sigma: 2.0000
[Saved fail dump] -> c:\Users\zihanx\Desktop\review\AC_car\merge_code\merge_final_version\fails_epoch0.npz | num_fails=62
  style 0: fails=11 (collision=11, timeout=0)
  style 1: fails=26 (collision=26, timeout=0)
  style 2: fails=25 (collision=25, timeout=0)
  [Style 0] Success=94.5%, Coll=5.5%, Speed=5.40
  [Style 1] Success=87.0%, Coll=13.0%, Speed=5.15
  [Style 2] Success=87.5%, Coll=12.5%, Speed=4.83
  [Overall] Success=89.7%, Coll=10.3%, Speed=5.13
>>> Saved BEST model (score=8.84) to: c:\Users\zihanx\Desktop\review\AC_car\AC_merge_

KeyboardInterrupt: 

In [53]:
import pandas as pd
import numpy as np
import re

CSV_PATH = "all_fails_summary.csv"   # 改成你的实际路径也行
E_PER_STYLE = 200                    # 你的固定集每个 style 有多少条（你日志里是 200）

df = pd.read_csv(CSV_PATH)

# 失败类型
df["fail_name"] = df["fail_type"].map({1: "collision", 2: "timeout"}).fillna("unknown")

# 从 file 字段提取 epoch（fails_epochXXX.npz）
def extract_epoch(s):
    m = re.search(r"fails_epoch(\d+)\.npz", str(s))
    return int(m.group(1)) if m else None

df["epoch"] = df["file"].apply(extract_epoch)

print("Rows:", len(df))
print("Epochs:", sorted(df["epoch"].dropna().unique().tolist()))
print("\n=== Count by style × fail_type (overall) ===")
tab = pd.crosstab(df["style"], df["fail_name"])
tab["total"] = tab.sum(axis=1)
print(tab)

print("\n=== Count by epoch × style × fail_type ===")
tab2 = df.groupby(["epoch", "style", "fail_name"]).size().unstack(fill_value=0)
tab2["total"] = tab2.sum(axis=1)
print(tab2.sort_index())

print("\n=== fail_step distribution (key diagnostic) ===")
print(df["fail_step"].value_counts().head(20))

# 把 collision 分成“step0 立即撞”和“step>0 后续撞”
col = df[df["fail_name"] == "collision"].copy()
col0 = col[col["fail_step"] == 0]
colL = col[col["fail_step"] > 0]

print("\n=== Collision split ===")
print("collision total:", len(col), " step0:", len(col0), " step>0:", len(colL))

def quick_stats(d, name):
    return pd.Series({
        "n": len(d),
        "s0_median": d["s0"].median(),
        "s0_p25": d["s0"].quantile(0.25),
        "s0_p75": d["s0"].quantile(0.75),
        "min_gap_med": d["min_gap_net"].median(),
        "min_gap_p25": d["min_gap_net"].quantile(0.25),
        "min_gap_<0%": (d["min_gap_net"] < 0).mean() * 100.0,
    }, name=name)

print(pd.concat([
    quick_stats(col0, "collision_step0"),
    quick_stats(colL, "collision_step>0"),
], axis=1))

# timeout 的 s0 分布（通常对应“起点太远+速度目标太保守”）
to = df[df["fail_name"] == "timeout"].copy()
print("\n=== Timeout s0 stats by style ===")
print(to.groupby("style")["s0"].agg(["count","mean","std","min","median","max"]))

# “永远失败”的场景：同一个 (style, ep) 在你保存的所有 epoch 都失败
df["sid"] = df["style"].astype(str) + "_" + df["ep"].astype(str)
n_epochs = df["epoch"].nunique()

fail_nuniq = df.groupby("sid")["epoch"].nunique()
persistent_sids = fail_nuniq[fail_nuniq == n_epochs].index
print(f"\n=== Persistent failing scenarios (fail in all {n_epochs} epochs dumps) ===")
print("persistent count:", len(persistent_sids), "out of total fixed:", 3 * E_PER_STYLE)

# persistent 的组成
pers = df[df["sid"].isin(persistent_sids)]
print(pd.crosstab(pers["style"], pers["fail_name"]))

# 建议输出给你做回放/重采样的“最危险”TopK（按 min_dist 最小）
print("\n=== Top dangerous fails per style (min_dist smallest) ===")
for sid, g in df.groupby("style"):
    worst = g.sort_values("min_dist", ascending=True).head(10)
    cols = ["epoch","style","ep","fail_name","fail_step","s0","min_dist","min_gap_net","dx_near","dv_near","v1_0"]
    print(f"\n--- style {sid} ---")
    print(worst[cols].to_string(index=False))


Rows: 207
Epochs: [0, 100, 200]

=== Count by style × fail_type (overall) ===
fail_name  collision  timeout  total
style                               
0                 33        0     33
1                 76        0     76
2                 84       14     98

=== Count by epoch × style × fail_type ===
fail_name    collision  timeout  total
epoch style                           
0     0             11        0     11
      1             26        0     26
      2             25        0     25
100   0             10        0     10
      1             23        0     23
      2             30        7     37
200   0             12        0     12
      1             27        0     27
      2             29        7     36

=== fail_step distribution (key diagnostic) ===
fail_step
0     139
99     14
6       9
3       7
5       6
2       6
1       6
4       4
7       4
12      3
13      3
11      1
22      1
18      1
10      1
34      1
16      1
Name: count, dtype: int64

=== Coll

In [ ]:
import os
import torch
import numpy as np
import torch.nn.functional as F

# ============================================================
# Warm Train Critic（只更新 Critic，Actor 冻结） + 漂移检测
# ============================================================
warm_epochs   = 500
N_per_style   = 4096
REWARD_SCALE  = 600.0

critic_lr = 3e-4
torch.nn.utils.clip_grad_norm_(critic.parameters(), 5.0)
optimizer_c = optim.Adam(critic.parameters(), lr=critic_lr)

# ---- 更稳的目标标准化：必须用“固定 per-style mu/std”，不能用 batch 自己的 ----
USE_PER_STYLE_NORM = True

# ---- probe 固定存档（数值型）----
PROBE_PATH = "fixed_probe_pack_REWARD600_K{}.npz".format(K)
fixed_batch_per_style = 512

# ---- 训练动作噪声 ----
NOISE_STD   = 0.03 * u_max     # 0.25 if u_max=5
NOISE_MIX_P = 0.2              # 50% noisy, 50% deterministic

# ---- 预估 per-style 统计量的 batch（建议 >= 8192）----
EST_BATCH = 8192
EPS_STD   = 1e-3

# 冻结 Actor
actor.eval()
for p in actor.parameters():
    p.requires_grad_(False)

critic.train()
for p in critic.parameters():
    p.requires_grad_(True)

print(f">>> Starting Critic Warm Training: {warm_epochs} epochs")
print(f">>> Actor: FROZEN (eval mode)")
print(f">>> Critic: TRAINING (lr={critic_lr:.2e})")
print(f">>> Probe file: {PROBE_PATH}")
print(f">>> REWARD_SCALE={REWARD_SCALE}, USE_PER_STYLE_NORM={USE_PER_STYLE_NORM}")
print(f">>> Noise std={NOISE_STD:.3f}, mix_p={NOISE_MIX_P:.2f}")

# ============================================================
# 0) 构造/加载 固定 probe batch  (NO object dtype)
# ============================================================
fixed_pack = []  # list[(sid, s_fix, a_fix, y_fix_raw)]

def to_torch(x_np, device):
    x_np = np.asarray(x_np, dtype=np.float32)
    return torch.from_numpy(x_np).to(device)

if os.path.exists(PROBE_PATH):
    data = np.load(PROBE_PATH)
    style_ids_np = data["style_ids"]              # [M]
    states_np    = data["states"]                 # [M,B,Ds]
    actions_np   = data["actions"]                # [M,B,Da]
    targets_np   = data["targets"]                # [M,B,1]   (raw target)

    for si in range(style_ids_np.shape[0]):
        sid   = int(style_ids_np[si])
        s_fix = to_torch(states_np[si], device)
        a_fix = to_torch(actions_np[si], device)
        y_fix = to_torch(targets_np[si], device)
        fixed_pack.append((sid, s_fix, a_fix, y_fix))

    print(">>> Loaded fixed probe pack from disk (numeric arrays).")

else:
    with torch.no_grad():
        M = len(style_list)
        B = fixed_batch_per_style

        # 先做一个 style，拿到维度，避免猜 shape
        style0 = style_list[0]
        gap_mode, base_mode = get_style_config(style0)
        ego_x0, ego_y0, v1_0, x_cars0, v_cars0 = get_mixed_initial_state_merge_Ncars(
            B, K, device, gap_mode=gap_mode, base_mode=base_mode
        )
        state0, style_ids0 = build_state_from_phys(ego_x0, ego_y0, v1_0, x_cars0, v_cars0, style0)
        a0 = torch.clamp(actor(state0), u_min, u_max)
        J0 = potentialFunction_update_merge_Ncars(
            a0, ego_x0, ego_y0, v1_0, x_cars0, v_cars0,
            style_ids0, ego_v_d, alpha, beta, delta, T, dt
        )
        y0 = (J0 / REWARD_SCALE).unsqueeze(1)

        Ds = state0.shape[1]
        Da = a0.shape[1]

        style_ids_np = np.array(style_list, dtype=np.int64)
        states_np  = np.zeros((M, B, Ds), dtype=np.float32)
        actions_np = np.zeros((M, B, Da), dtype=np.float32)
        targets_np = np.zeros((M, B, 1),  dtype=np.float32)

        # 填第一个
        states_np[0]  = state0.detach().cpu().numpy().astype(np.float32)
        actions_np[0] = a0.detach().cpu().numpy().astype(np.float32)
        targets_np[0] = y0.detach().cpu().numpy().astype(np.float32)
        fixed_pack.append((int(style0), state0, a0, y0))

        # 其余 style
        for si, style_id in enumerate(style_list[1:], start=1):
            gap_mode, base_mode = get_style_config(style_id)
            ego_x0, ego_y0, v1_0, x_cars0, v_cars0 = get_mixed_initial_state_merge_Ncars(
                B, K, device, gap_mode=gap_mode, base_mode=base_mode
            )
            state_i, style_ids_i = build_state_from_phys(
                ego_x0, ego_y0, v1_0, x_cars0, v_cars0, style_id
            )
            a_i = torch.clamp(actor(state_i), u_min, u_max)
            J_i = potentialFunction_update_merge_Ncars(
                a_i, ego_x0, ego_y0, v1_0, x_cars0, v_cars0,
                style_ids_i, ego_v_d, alpha, beta, delta, T, dt
            )
            y_i = (J_i / REWARD_SCALE).unsqueeze(1)

            assert state_i.shape[1] == Ds, (state_i.shape, Ds)
            assert a_i.shape[1] == Da, (a_i.shape, Da)

            states_np[si]  = state_i.detach().cpu().numpy().astype(np.float32)
            actions_np[si] = a_i.detach().cpu().numpy().astype(np.float32)
            targets_np[si] = y_i.detach().cpu().numpy().astype(np.float32)

            fixed_pack.append((int(style_id), state_i, a_i, y_i))

        np.savez(PROBE_PATH, style_ids=style_ids_np, states=states_np, actions=actions_np, targets=targets_np)

    print(">>> Fixed probe pack prepared and saved to disk (numeric arrays).")


# ============================================================
# 0.5) 预估每个 style 的固定 (mu,std) —— 训练与 probe 都用它
# ============================================================
mu_style  = {}
std_style = {}

with torch.no_grad():
    for sid in style_list:
        gap_mode, base_mode = get_style_config(sid)
        ego_x0, ego_y0, v1_0, x_cars0, v_cars0 = get_mixed_initial_state_merge_Ncars(
            EST_BATCH, K, device, gap_mode=gap_mode, base_mode=base_mode
        )
        s0, style_ids = build_state_from_phys(ego_x0, ego_y0, v1_0, x_cars0, v_cars0, sid)
        a_det = torch.clamp(actor(s0), u_min, u_max)
        noise = torch.randn_like(a_det) * NOISE_STD
        a_noi = torch.clamp(a_det + noise, u_min, u_max)
        mix = (torch.rand_like(a_det) < NOISE_MIX_P)
        a0 = torch.where(mix, a_det, a_noi)


        J = potentialFunction_update_merge_Ncars(
            a0, ego_x0, ego_y0, v1_0, x_cars0, v_cars0,
            style_ids, ego_v_d, alpha, beta, delta, T, dt
        )
        y = (J / REWARD_SCALE).unsqueeze(1)  # raw y

        mu = y.mean(dim=0, keepdim=True)
        sd = y.std(dim=0, keepdim=True, unbiased=False).clamp_min(EPS_STD)

        mu_style[int(sid)]  = mu
        std_style[int(sid)] = sd

print(">>> Fixed per-style mu/std prepared:")
for sid in style_list:
    m = float(mu_style[int(sid)].item())
    s = float(std_style[int(sid)].item())
    print(f"    style {sid}: mu={m:.4f}, std={s:.4f}")


# ============================================================
# 1) Warm train loop
# ============================================================
for epoch in range(warm_epochs):
    total_critic_loss = torch.tensor(0.0, device=device)

    for style_id in style_list:
        gap_mode, base_mode = get_style_config(style_id)

        ego_x0, ego_y0, v1_0, x_cars0, v_cars0 = get_mixed_initial_state_merge_Ncars(
            N_per_style, K, device,
            gap_mode=gap_mode,
            base_mode=base_mode,
        )

        state0, style_ids = build_state_from_phys(
            ego_x0, ego_y0, v1_0,
            x_cars0, v_cars0, style_id
        )

        with torch.no_grad():
            a_det = torch.clamp(actor(state0), u_min, u_max)
            noise = torch.randn_like(a_det) * NOISE_STD
            a_noi = torch.clamp(a_det + noise, u_min, u_max)
            mix = (torch.rand_like(a_det) < NOISE_MIX_P)
            a_train = torch.where(mix, a_det, a_noi)

            J_tot = potentialFunction_update_merge_Ncars(
                a_train, ego_x0, ego_y0, v1_0, x_cars0, v_cars0,
                style_ids, ego_v_d, alpha, beta, delta, T, dt
            )
            y = (J_tot / REWARD_SCALE).unsqueeze(1)  # raw target

            if USE_PER_STYLE_NORM:
                mu = mu_style[int(style_id)]
                sd = std_style[int(style_id)]
                y_train = (y - mu) / sd
            else:
                y_train = y

        Q_pred = critic(state0, a_train)
        loss = F.mse_loss(Q_pred, y_train)

        total_critic_loss = total_critic_loss + loss

    total_critic_loss = total_critic_loss / M_styles

    optimizer_c.zero_grad()
    total_critic_loss.backward()
    torch.nn.utils.clip_grad_norm_(critic.parameters(), 1.0)
    optimizer_c.step()

    # =========================================================
    # 2) 每 50 epoch：probe 打印（必须在同尺度下比较）
    # =========================================================
    if epoch % 50 == 0:
        with torch.no_grad():
            print(f"\n[Warm {epoch:4d}/{warm_epochs}] train_loss={total_critic_loss.item():.4f}")
            for (sid, s_fix, a_fix, y_fix_raw) in fixed_pack:
                q_fix = critic(s_fix, a_fix)

                if USE_PER_STYLE_NORM:
                    mu = mu_style[int(sid)]
                    sd = std_style[int(sid)]
                    y_cmp = (y_fix_raw - mu) / sd
                else:
                    y_cmp = y_fix_raw

                mse = float(((q_fix - y_cmp) ** 2).mean().item())
                hub = float(F.smooth_l1_loss(q_fix, y_cmp, beta=1.0).item())

                q_mean = float(q_fix.mean().item())
                q_std  = float(q_fix.std(unbiased=False).item())
                y_mean = float(y_cmp.mean().item())
                y_std  = float(y_cmp.std(unbiased=False).item())
                corr = torch.corrcoef(torch.stack([q_fix.squeeze(), y_cmp.squeeze()]))[0,1].item()

                print(
                    f"  probe style {sid}: "
                    f"Q_mean={q_mean:.4f}, Q_std={q_std:.4f}, "
                    f"Y_mean={y_mean:.4f}, Y_std={y_std:.4f}, "
                    f"Huber={hub:.4f}, MSE={mse:.4f}"
                    
                )
                print(f"... corr={corr:.3f}")
print(">>> Critic Warm Training Complete!")


In [ ]:
with np.load(PROBE_PATH, allow_pickle=True) as data:
    style_ids = data["style_ids"]
    states = data["states"]
# 出了 with，文件就释放了


In [ ]:
import gc, os
gc.collect()
os.remove(PROBE_PATH)
